<!-- notebook-header -->
<a id="top"></a>

# 4. Stacks, Queues & Linked Lists

*LeetCode Playbook · notebook 4 of 12* · The linear structures: what waits on a stack, what a monotonic stack resolves, and pointer rewiring.

**In this notebook:** [Stacks & Queues](#s07) · [Monotonic Stack](#s08) · [Linked Lists](#s10)

← [Two Pointers, Sliding Window & Strings](03_Two_Pointers_Sliding_Window_Strings.ipynb) · [All notebooks](01_Start_Here.ipynb#notebooks) · [A-Z problem finder](12_Interview_Day.ipynb#finder) · [Binary Search & Sorting](05_Binary_Search_Sorting.ipynb) →

Run the setup cell below first; after that every section runs from its first cell.

In [ ]:
# Setup: everything this notebook imports. Run this cell first.
from collections import Counter, defaultdict, deque, OrderedDict
from functools import cmp_to_key, lru_cache
from itertools import accumulate, combinations, permutations, product
import bisect, heapq, math, random, string

<a id="s07"></a>

## Stacks & Queues

> A **stack** is a pile of unfinished business: whatever you opened last must be finished first, so it sits on top (LIFO). A **queue** is a line: whoever arrived first is served, or expires, first (FIFO). To pick one, ask a single question: *in what order do my pending items get finished?*

**Reach for it when** you see nesting or matching: brackets, `k[...]`, the parentheses of an expression. The same goes for "the most recent unmatched thing", for going back with `..` in a path or with undo, for evaluating an expression, and for a newcomer that only ever meets the nearest survivor, as in a collision. All of those are a **stack**.

When items are handled in arrival order, because the oldest expires first, players take turns or a buffer has a fixed size, that is a **queue**, written with `collections.deque`. The most common queue of all is BFS: "nearest first", "fewest steps" or "level by level" means the frontier is a deque, and that one lives in [Graphs I](09_Graphs.ipynb#s17).

**In this repo:** `stack/` (10 of its 16 problems; the "next greater" family is in [Monotonic Stack](#s08)) · `queues/` (6 problems) · bank: `practice/simple/13_valid_parentheses.py`, `practice/simple/14_min_stack.py`, `practice/simple/15_evaluate_reverse_polish_notation.py` · basics in `practice/simple/basics/stacks/`: `01_array_stack_and_queue_via_two_stacks.py`, `02_simplify_unix_path.py`, `03_decode_string.py`, `04_basic_calculator_ii.py`, `05_asteroid_collision.py`

### The picture

```text
s = { [ ( ) ] ( }                        the stack after each character (top on the right)

    {   opener: push                     {
    [   opener: push                     { [
    (   opener: push                     { [ (
    )   top is ( : a match, pop          { [
    ]   top is [ : a match, pop          {
    (   opener: push                     { (
    }   top is ( but } needs {           invalid: the innermost open bracket is (
```

```text
stack (LIFO):  [ a  b  c  d ]  <- push and pop here, at the top (d is the most recent)
queue (FIFO):  append -> [ e  d  c  b  a ] -> popleft         (a is the oldest, it leaves first)
```

**Why it is fast:** the brute force for nesting finds an innermost pair like `()`, deletes it, and rescans from the start: O(n²). A stack holds exactly the unfinished items in the order they were opened, so each character is pushed once and popped at most once: O(n).

Queues have the same story. `list.pop(0)` shifts every remaining item, O(n) per call. A `deque` moves nothing, and a ring buffer moves an index instead of the data, so with either one an item leaves the line in O(1).

**Why it is correct:** brackets nest, so a bracket opened later must be closed earlier. The next closer can therefore only match the innermost open bracket, which is the most recent one: the top. A closer that does not match the top can never be matched by anything later, so failing at once is safe.

### From idea to code

**The idea in one sentence:** *walk the input once; push whatever is still unfinished; when an item finishes something, the thing it finishes is on top, so pop it and combine.*

The seven decisions, for a stack first. The **State** is a Python list used as a stack of *unfinished* items, and its **Definition** goes in a comment: `stack = openers not closed yet, innermost on top`. The **Invariant** is that after reading `s[:i]` the stack holds exactly the unfinished items of that prefix, most recent on top. A **Step** pushes an opener or a value; a closer or an operator pops what it finishes, combines, and pushes the result when there is one.

The **Fix** is a `while` loop, needed only when one newcomer can finish several items in a row, as in a collision or a run that vanishes: pop while the newcomer beats the top. **Record** at the pop whenever the pop measures something: the length of a well-formed run in Longest Valid Parentheses, a distance or an area in [Monotonic Stack](#s08). Otherwise the answer is what is left at the end, and an early `return False` records a failure the moment it is certain.

**Init** is `stack = []` and `num = 0`, sometimes with a sentinel: a `-1` barrier under the indices, a trailing `"+"` on the input. **Return** `not stack` for matching, the single value left for evaluation, and the stack itself when the survivors are the answer.

A queue makes the same decisions with the ends swapped. The state is a `deque` in arrival order, defined as `q = pings in [t - 3000, t], oldest at the front`, and the invariant is that it holds exactly the live items. A step appends the new item at the back, the fix pops the front while it has expired, and the record, after the fix, is `len(q)` or the front. Init is `q = deque()`; the return is the size or the front per call, or whoever is left at the end.

One more decision hides inside the state: what goes in one stack entry. The answer is exactly what you will need at the moment it is popped. It is the easiest decision to skip, and the one that makes the rest of the code obvious.

Valid Parentheses (20), which asks whether brackets are properly nested, pushes the opener itself, because the closer must match its type. Evaluate Reverse Polish Notation (150), which evaluates a postfix expression, pushes values, because an operator combines the two newest. Basic Calculator II (227), which evaluates `"3+2*2"` with the usual precedence, pushes signed terms: `+` and `-` wait for the final sum, while `*` and `/` edit the top. Decode String (394), which expands `3[a]` into `aaa`, parks a pair, the text before `[` and the count, because `]` glues prefix + inner × count.

Longest Valid Parentheses (32), which wants the longest well-formed stretch of `(` and `)`, pushes an index on top of a `-1` barrier, because a length is `i - stack[-1]`. Asteroid Collision (735), where the smaller of two colliding asteroids explodes, pushes a survivor, because the newcomer fights the nearest survivor. Min Stack (155), a stack that also answers `getMin` in O(1), pushes the value together with the minimum at or below it, because after a pop the minimum must already be known.

Remove All Adjacent Duplicates in String II (1209), which deletes every run of k equal letters, pushes `[char, run length]`, because a run can grow again after the run above it vanishes. Simplify Path (71), which cleans up a Unix path, pushes a directory name, because `..` undoes the most recent one. If you cannot say what you will need at the pop, solve two pops by hand first.

Two stack templates come first, one that pops to *match* and one that pops to *combine*. Valid Parentheses asks whether a string of `()[]{}` is properly nested: `"{[()]}"` is, `"([)]"` is not. Walk the string. An opener is unfinished business, `stack.append(ch)`. A closer must finish the most recent opener, `stack[-1]`, which you may read only after checking that the stack is not empty; `stack.pop()` finishes it. At the end, `return not stack` asks whether anything is still open.

Evaluate Reverse Polish Notation asks for the value of a postfix expression: `["2", "1", "+", "3", "*"]` is `(2 + 1) * 3 = 9`. Numbers wait on the stack, and an operator combines the two newest, with the right operand coming off first: `b = stack.pop()`, then `a = stack.pop()`, then `a - b`. Order matters in both templates. `is_valid` looks at the top *before* it pops, because the check decides whether the pop is legal, and `eval_rpn` pops `b` before `a`, because the right operand was pushed last.

In [ ]:
def is_valid(s):
    partner = {")": "(", "]": "[", "}": "{"}     # closer -> its opener
    stack = []                               # STATE + INIT: openers not closed yet, innermost on top
    for ch in s:
        if ch not in partner:
            stack.append(ch)                 # STEP (opener): one more unfinished bracket
        elif not stack or stack[-1] != partner[ch]:
            return False                     # RECORD: the answer is known now: invalid
        else:
            stack.pop()                      # STEP (closer): it finishes the innermost opener
    return not stack                         # RETURN: valid iff nothing was left open


OPS = {"+": lambda a, b: a + b, "-": lambda a, b: a - b,
       "*": lambda a, b: a * b, "/": lambda a, b: int(a / b)}   # int(a / b) truncates toward 0


def eval_rpn(tokens):
    stack = []                               # STATE + INIT: values waiting for an operator
    for tok in tokens:
        if tok in OPS:
            b = stack.pop()                  # the RIGHT operand was pushed last
            a = stack.pop()
            stack.append(OPS[tok](a, b))     # STEP (operator): two waiting values become one
        else:
            stack.append(int(tok))           # STEP (number): it waits; "-3" is a number too
    return stack[-1]                         # RETURN: exactly one value is left


print(is_valid("{[()]}"), is_valid("([)]"), is_valid("(("))                        # True False False
print(eval_rpn(["2", "1", "+", "3", "*"]), eval_rpn(["4", "13", "5", "/", "+"]))   # 9 6

**Try it**
- Delete `not stack or` from the `elif` and run `is_valid("]")`: `IndexError`, because the code peeks at the top of an empty stack.
- Replace `return not stack` with `return True` and run `is_valid("((")`: it says `True`, although nothing was ever closed.
- Swap the two pops (pop `a` first) and run `eval_rpn(["6", "3", "-"])`: you get -3 instead of 3, because the operands come off the stack in reverse order.
- Change `int(a / b)` to `a // b` in `OPS` and run `eval_rpn(["7", "-3", "/"])`: -3 instead of -2, because `//` floors and the problem truncates toward zero.

A queue template has the same shape, but items leave from the *other* end: "join the line" is `q.append(x)`, "leave the line" is `q.popleft()`, never `list.pop(0)`, and "the oldest item" is `q[0]`. When items expire in the order they arrived, only the front ever needs checking. The lines then run in the order STEP, FIX, RECORD: join, drop whatever has expired, and measure last, so that only live items are counted.

Number of Recent Calls (933) asks `ping(t)` to count the pings in `[t - 3000, t]`, and `t` only grows: pings at 1, 100, 3001 and 3002 answer 1, 2, 3 and 3, because by 3002 the ping at 1 has expired. Each ping joins at the back, the front leaves while it is older than `t - 3000`, and the answer is the length of what is left. Appending first also means the FIX loop can never empty the deque, because `t` itself never expires, so it needs no `self.q and` guard.

In [ ]:
class RecentCounter:                         # 933: how many pings in [t - 3000, t]?
    def __init__(self):
        self.q = deque()                     # STATE + INIT: live pings, oldest at the front

    def ping(self, t):                       # t only grows
        self.q.append(t)                     # STEP: the newest joins at the back
        while self.q[0] < t - 3000:          # FIX: drop expired pings (q never empties: t is in it)
            self.q.popleft()
        return len(self.q)                   # RECORD + RETURN: the window is the deque


counter = RecentCounter()
print([counter.ping(t) for t in [1, 100, 3001, 3002]])   # [1, 2, 3, 3]

**Try it**
- Change `<` to `<=` in the `while` and replay `[1, 100, 3001, 3002]`: the third answer becomes 2, because the ping at `t = 1` is exactly 3000 ms old and still belongs to `[1, 3001]`.
- Change the `while` to `if` and ping `[1, 2, 3, 5000]`: the last answer is 3 instead of 1, since only one expired ping can leave per call.
- Replace `3000` with `10` and predict `[1, 5, 11, 12, 30]` before running: `[1, 2, 3, 3, 1]`.

### Watch it work

The stack after every token of `["4", "13", "5", "/", "+"]`, which is `4 + 13 / 5` and evaluates to 6. Notice that `/` takes `13` and `5`, the two most recent values, and never touches the `4` underneath: that is why postfix needs no parentheses.

In [ ]:
def trace_rpn(tokens):
    stack = []
    for tok in tokens:
        note = ""
        if tok in OPS:
            b, a = stack.pop(), stack.pop()
            stack.append(OPS[tok](a, b))
            note = f"{a} {tok} {b} = {stack[-1]}"
        else:
            stack.append(int(tok))
        print(f"{tok:>3}   stack = {str(stack):<17} {note}".rstrip())


trace_rpn(["4", "13", "5", "/", "+"])

**Try it**
- Run `trace_rpn(["2", "1", "+", "3", "*"])` and predict the stack just before the `*` (it is `[3, 3]`).
- Trace `["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]`: the stack never holds more than 4 values, `6 / -132` truncates to 0, and the result is 22.
- Feed a broken expression, `trace_rpn(["1", "+"])`: the second `pop` raises `IndexError`. A defensive version checks `len(stack) >= 2` before popping.

### Where it goes wrong

1. **Peeking at an empty stack.** A closer with nothing open (`"]"`, `"())"`) makes `stack[-1]` raise `IndexError`. Check `not stack` first, in the same condition.
2. **Forgetting the leftovers.** `"(("` never fails inside the loop. The answer is `not stack`, not `True`.
3. **Counting instead of matching.** `"([)]"` has balanced counts but is invalid. A plain counter is enough only with a single bracket type, as in Minimum Add to Make Parentheses Valid (921), which asks for the fewest brackets to insert: then every stack entry is the same `(` and only the height matters.
4. **Operand order.** The first pop is the *right* operand: `["6", "3", "-"]` is `6 - 3 = 3`, not -3.
5. **Floor instead of truncate.** Python's `//` floors (`7 // -3 == -3`), but these problems truncate toward zero (`int(7 / -3) == -2`). It also bites calculators that store `-3` as a signed term: `"14-3/2"` must be 13, and `-3 // 2` would make it 12.
6. **The last number is never applied.** A calculator applies an operator when the *next* operator arrives, so the final number needs a flush: loop over `s + "+"`. Without the sentinel, `"3+2*2"` gives 5. Testing `i == len(s) - 1` also works, but only in an `if` of its own: chained as an `elif` after the digit test, `"42"` gives 0.
7. **`if` instead of `while` for collisions.** A left-mover can destroy several right-movers in a row: `[1, 2, -5]` must end as `[-5]`, not `[1, -5]`.
8. **`list.pop(0)` as a queue.** Each call shifts every remaining item, so n dequeues cost about n²/2 moves. Use `deque.popleft()`, or a ring buffer that moves `head` instead of the data.
9. **An expiry loop without `q and`.** It is safe in `ping` only because `t` was just appended. A read-only method has no such guarantee. Design Hit Counter (362) asks `getHits(t)` for the hits of the last 300 seconds; [From Idea to Code](01_Start_Here.ipynb#s01) has the base version and [Design Problems](11_Design.ipynb#s24) the follow-up. Write `while q and q[0] <= t - 300`, or a call on an empty counter raises `IndexError`.

### Edge cases to say out loud

Empty input · a lone closer · only openers · the wrong type innermost (`"([)]"`) · a single RPN token · negative numbers (`"-3"` is a number, not an operator) · dividing a negative · multi-digit numbers · a ping exactly 3000 ms old (still counts). The cell checks each of them against the three templates above.

In [ ]:
assert is_valid("") is True
assert is_valid("]") is False                         # closer with nothing open
assert is_valid("((") is False                        # leftovers
assert is_valid("([)]") is False                      # counts match, order doesn't
assert eval_rpn(["5"]) == 5
assert eval_rpn(["7", "-3", "/"]) == -2               # "-3" is a number; truncate toward 0
assert eval_rpn(["6", "3", "-"]) == 3                 # a - b, with b popped first
assert eval_rpn(["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]) == 22
c = RecentCounter()
assert [c.ping(t) for t in [1, 3001, 3002]] == [1, 2, 2]   # t = 1 still counts at 3001
print("edge cases pass")

**Try it**
- Predict, then add: `assert is_valid("(]") is False` (the top is `(`, but `]` needs `[`).
- Write the assert for `eval_rpn(["3", "4", "-", "5", "*"])` before running it: `(3 - 4) * 5 = -5`.
- Swap the stack for one counter (+1 on an opener, −1 on a closer, fail below 0) and run `"([)]"`: it passes as valid. That is trap 3 in one line.

### Variations

| Variation | What changes from the template | Problems |
|---|---|---|
| **Match pairs** | push openers; a closer must equal the top | Valid Parentheses (20) |
| **Stack collapses to a counter** | one bracket type: keep only the height; a `)` at height 0 costs one insertion | Minimum Add to Make Parentheses Valid (921): the fewest brackets to insert |
| **Postfix evaluation** | numbers push; an operator pops two and pushes one | Evaluate Reverse Polish Notation (150) |
| **Precedence** | push signed terms; `*` and `/` edit the top term; the answer is `sum(stack)` | Basic Calculator II (227) |
| **Park and resume** | on an opener, push the current context and start fresh; on a closer, pop it and merge | Decode String (394) |
| **Mark, then rebuild** | a stack of indices of unmatched `(`; drop every marked character | Minimum Remove to Make Valid Parentheses (1249): delete the fewest brackets, in [Strings](03_Two_Pointers_Sliding_Window_Strings.ipynb#s20) |
| **Undo / go up** | names push, `..` pops (if anything is there), `.` and `""` do nothing | Simplify Path (71) |
| **Collisions** | the newcomer fights the top *while* they collide | Asteroid Collision (735) |
| **Stack of runs** | entry `[char, count]`; k in a row vanish, and the run below may grow again | Remove All Adjacent Duplicates in String II (1209) and I (1047): `"abbaca"` → `"ca"` |
| **Remember the minimum** | each entry stores `(value, min of everything at or below it)` | Min Stack (155) |
| **Queue from two stacks** | inbox takes pushes, outbox serves pops; refill only when the outbox is empty | Implement Queue using Stacks (232) |
| **Stack from one queue** | after each push, rotate the older items behind the new one (`len - 1` times) | Implement Stack using Queues (225): push, pop, top and empty from queue moves only |
| **Ring buffer** | fixed array + `head` + `count`; every index goes through `% k` | Design Circular Queue (622) and Design Circular Deque (641): a bounded queue in a fixed array |
| **Time window** | deque of times; pop the front while it is too old | Number of Recent Calls (933); Design Hit Counter (362) |
| **Round robin** | one queue per side; the earlier front acts and rejoins as `index + n` | Dota2 Senate (649): which party wins a ban war |
| *Second pass:* **Index stack with a barrier** | push indices, seeded with `-1`; after a match the valid run is `i - stack[-1]` | Longest Valid Parentheses (32) |
| *Second pass:* **Collapse at the closer** | push everything; at `)` pop the group's operands, then its operator, and push one result | Parsing a Boolean Expression (1106): evaluate and/or/not groups over `t` and `f` |
| *Second pass:* **Park and resume, with arithmetic** | at `(` park the terms so far and the pending operator; at `)` the group becomes one number | Basic Calculator (224) and Basic Calculator III (772): expressions with parentheses |

The code below covers the five shapes that come up most: precedence, park and resume, collisions and undo, entries that remember, and a queue from two stacks. The ring buffer and the round robin need only their idea.

A ring buffer, as in Design Circular Queue (622), moves a label instead of the data: the items are the `count` slots that start at `head` in a fixed array of k, and every index goes through `% k`. Design Circular Deque (641) adds the front end by stepping `head` back, `(head - 1) % k`, before it writes, and self-check 4 shows why `count` is kept.

The round robin of Dota2 Senate (649) gives each party a queue of turn indices. The earlier front bans the other front and rejoins its own queue as `index + n`, one full round later, so both queues stay in turn order and the party left standing wins.

Precedence comes first because it is Evaluate RPN with the operators in their usual place. Basic Calculator II (227) asks for the value of a string such as `"3+2*2"`, which is 7: `*` and `/` bind before `+` and `-`, and division truncates toward zero, so `" 3/2 "` is 1. An operator cannot be applied when you read it, because its right operand has not been read yet.

So the operator waits *in front of* the current number, in `op`, and is applied when the next operator arrives. `+` and `-` are put off by pushing signed terms, which the end adds up; `*` and `/` bind tighter, so they combine with the term on top at once. The number is read digit by digit, `num = num * 10 + int(ch)`, and a sentinel `"+"` at the end makes the last number get applied too.

In [ ]:
def push_term(stack, op, num):               # put num on the stack the way op says
    if op == "+":
        stack.append(num)
    elif op == "-":
        stack.append(-num)                   # the sign is stored with the term
    elif op == "*":
        stack.append(stack.pop() * num)      # binds to the term before it
    else:
        stack.append(int(stack.pop() / num)) # -3 / 2 -> -1 (truncate), not -2


def calculate_ii(s):
    stack, num, op = [], 0, "+"              # STATE + INIT: signed terms; op = the operator in front of num
    for ch in s + "+":                       # the sentinel "+" flushes the last number
        if ch.isdigit():
            num = num * 10 + int(ch)         # one more digit
        elif ch in "+-*/":
            push_term(stack, op, num)        # STEP: num is complete, apply the operator in front of it
            num, op = 0, ch
    return sum(stack)                        # RETURN: the postponed + and - happen here


print(calculate_ii("3+2*2"), calculate_ii(" 3/2 "), calculate_ii("14-3/2"))   # 7 1 13

**Try it**
- In `push_term`, change `int(stack.pop() / num)` to `stack.pop() // num` and run `calculate_ii("14-3/2")`: 12 instead of 13, because the term is stored as `-3` and `-3 // 2` floors to -2.
- Delete `+ "+"` from the loop and run `calculate_ii("3+2*2")`: 5 instead of 7. The pending `* 2` is never applied.
- Print `stack` just before the `return` for `"2*3+4*5-6/2"`: `[6, 20, -3]`, one signed term per `+` or `-`, which sum to 23.

Park and resume comes next because the thing on the stack is no longer a value but a whole context. Decode String (394) asks you to expand `k[text]`, which may nest: `"3[a2[c]]"` is `"accaccacc"`. An opening bracket means *save where I am and start fresh*; a closing bracket means *finish the inner part, restore what was saved, and merge*. The stack parks `(text so far, repeat count)` at every `[`, and `cur` is always the text of the innermost open group.

In [ ]:
def decode_string(s):
    stack, cur, k = [], "", 0                # STATE + INIT: parked (text before '[', count); cur = open group's text
    for ch in s:
        if ch.isdigit():
            k = k * 10 + int(ch)             # "10[a]": counts can have several digits
        elif ch == "[":
            stack.append((cur, k))           # STEP: park, then start fresh inside
            cur, k = "", 0
        elif ch == "]":
            prev, count = stack.pop()        # STEP: resume the parked context
            cur = prev + cur * count
        else:
            cur += ch                        # STEP: a letter extends the open group
    return cur                               # RETURN


print(decode_string("3[a2[c]]"), decode_string("2[abc]3[cd]ef"))   # accaccacc abcabccdcdcdef

**Try it**
- Delete the line `cur, k = "", 0` and run `decode_string("a2[b]")`: `aabab` instead of `abb`. The parked prefix `"a"` was also left in the inner text, so it got repeated.
- Change `k = k * 10 + int(ch)` to `k = int(ch)` and run `decode_string("10[a]")`: an empty string, because only the last digit (0) survived.
- Print `stack` right after each push for `"3[a2[c]]"`: `[('', 3)]`, then `[('', 3), ('a', 2)]`. One frame per open bracket.
- Predict `decode_string("2[a]b")` before running: `aab`. Text after a group just extends `cur`.

The next two problems pop for a different reason, to destroy and to undo. Asteroid Collision (735) asks which asteroids survive. The size is the absolute value, the sign is the direction, and when two meet the smaller explodes, both if they are equal: `[5, 10, -5]` ends as `[5, 10]`, `[8, -8]` as `[]`. Only a left-mover arriving after a right-mover can crash, and it meets the most recent survivor first. So it fights the top *while* it keeps winning (FIX), and it is pushed only if it survives (STEP).

Simplify Path (71) asks for the canonical form of a Unix path: `"/a/./b/../../c/"` is `"/c"`. A path is a stack of directory names: a name goes one level down, `..` goes one level up, but never above the root. The same function is in `practice/simple/basics/stacks/02_simplify_unix_path.py`.

In [ ]:
def asteroid_collision(asteroids):
    stack = []                               # STATE + INIT: survivors so far, left to right
    for a in asteroids:
        alive = True
        while alive and a < 0 and stack and stack[-1] > 0:   # FIX: only -> <- collide
            if stack[-1] < -a:
                stack.pop()                  # the top explodes, a flies on
            else:
                if stack[-1] == -a:
                    stack.pop()              # same size: both explode
                alive = False
        if alive:
            stack.append(a)                  # STEP: a survives (for now)
    return stack                             # RETURN: the survivors


def simplify_path(path):
    stack = []                               # STATE + INIT: directory names, the root side at the bottom
    for part in path.split("/"):
        if part == "..":
            if stack:                        # never above the root
                stack.pop()                  # STEP: one level up
        elif part and part != ".":           # "" (from "//") and "." change nothing
            stack.append(part)               # STEP: one level down
    return "/" + "/".join(stack)             # RETURN


print(asteroid_collision([5, 10, -5]), asteroid_collision([8, -8]), asteroid_collision([1, 2, -5]))   # [5, 10] [] [-5]
print(simplify_path("/a/./b/../../c/"), simplify_path("/../"), simplify_path("/home//foo/"))   # /c / /home/foo

**Try it**
- Change the `while` in `asteroid_collision` to `if` and run `[1, 2, -5]`: `[1, -5]` instead of `[-5]`. One fight per asteroid is not enough.
- Delete the two lines of the same-size case and run `[8, -8]`: `[8]` instead of `[]`.
- Run `asteroid_collision([-2, -1, 1, 2])`: nothing collides. Left-movers already on the left never meet the right-movers behind them.
- Remove the `if stack:` guard and run `simplify_path("/../")`: `IndexError`, because `..` at the root tries to pop an empty stack.

Entries that remember come next, because here the decision "what goes in one entry" is the whole solution. Min Stack (155) asks for a stack whose `push`, `pop`, `top` and `getMin` all run in O(1): push −2, 0 and −3, and `getMin` is −3; pop, and it is −2 again. Remove All Adjacent Duplicates in String II (1209) asks you to delete every run of k equal neighbours, again and again, until none is left: `"deeedbbcccbdaa"` with k = 3 becomes `"aa"`.

When the pop must know something about what lies *below* the top, store it in the entry. A stack only changes at its top, so the minimum of everything below an entry cannot change while that entry is there: Min Stack stores it next to the value. Remove Adjacent Duplicates II stores a run as `[char, count]`, because when a run of k vanishes, the run underneath can keep growing.

In [ ]:
class MinStack:                              # 155
    def __init__(self):
        self.stack = []                      # STATE + INIT: (value, min of this entry and everything below)

    def push(self, x):
        low = min(x, self.stack[-1][1]) if self.stack else x
        self.stack.append((x, low))          # STEP: the entry carries its own minimum

    def pop(self):
        self.stack.pop()                     # the entry below still knows its own minimum

    def top(self):
        return self.stack[-1][0]

    def getMin(self):
        return self.stack[-1][1]


def remove_duplicates(s, k):                 # 1209
    stack = []                               # STATE + INIT: [char, how many in a row]
    for ch in s:
        if stack and stack[-1][0] == ch:
            stack[-1][1] += 1                # STEP: the run on top grows
            if stack[-1][1] == k:
                stack.pop()                  # FIX: k in a row vanish; the run below may grow again
        else:
            stack.append([ch, 1])            # STEP: a new run starts
    return "".join(ch * n for ch, n in stack)    # RETURN


ms = MinStack()
for x in [-2, 0, -3]:
    ms.push(x)
before = ms.getMin()
ms.pop()
print(before, ms.top(), ms.getMin())                                           # -3 0 -2
print(remove_duplicates("deeedbbcccbdaa", 3), remove_duplicates("abbaca", 2))   # aa ca

**Try it**
- Print `ms.stack` after the three pushes: `[(-2, -2), (0, -2), (-3, -3)]`. The right column is the running minimum, frozen at each height.
- Build the two-stack version instead: a `mins` stack pushed only when `x < mins[-1]`, popped when the popped value equals `mins[-1]`. Push 0, push 0, pop, then `getMin`: `IndexError`, because the single 0 in `mins` was shared by both pushes. Push when `x <= mins[-1]`.
- Print `stack` after every character of `"deeedbbcccbdaa"` with k = 3: when `ccc` vanishes, the `bb` underneath meets one more `b` and vanishes too, and then `ddd` does the same.
- Make the entry a tuple, `(ch, 1)`: `TypeError` on `stack[-1][1] += 1`, because a tuple can't change. That is why the entry is a list.

A queue from two stacks closes the main path, because it shows that a stack and a queue differ by exactly one reversal. Implement Queue using Stacks (232) asks for a FIFO queue with `push`, `pop`, `peek` and `empty`, built from stack moves only: push 1, push 2, and `peek` is 1. Pouring one stack into another reverses it, which turns "newest on top" into "oldest on top". Pour only when the outbox is empty, so every item is poured once: amortised O(1), which means the cost averaged over all the operations.

In [ ]:
class MyQueue:                               # 232
    def __init__(self):
        self.inbox, self.outbox = [], []     # STATE + INIT: inbox newest on top; outbox oldest on top

    def push(self, x):
        self.inbox.append(x)                 # STEP

    def peek(self):
        if not self.outbox:                  # FIX: refill only when empty, or the order breaks
            while self.inbox:
                self.outbox.append(self.inbox.pop())
        return self.outbox[-1]               # RETURN: the oldest item

    def pop(self):
        self.peek()                          # make sure the oldest item is on the outbox top
        return self.outbox.pop()

    def empty(self):
        return not self.inbox and not self.outbox


q = MyQueue()
q.push(1)
q.push(2)
first = q.pop()
q.push(3)                                    # 3 must wait behind 2
print(first, q.pop(), q.pop(), q.empty())    # 1 2 3 True

**Try it**
- In `peek`, remove `if not self.outbox:` so it always pours, and replay the lines above: the second `pop` returns 3 instead of 2, because 3 was poured on top of the waiting 2.
- Print `q.inbox, q.outbox` after every call: each item crosses from inbox to outbox exactly once.
- Push 1 to 1000, then pop all 1000: count the appends to `outbox`. There are exactly 1000, so the expensive pour is paid once per item.

The rest of this section is a second pass: Hard problems that reuse the same moves. Skip them until the main path is automatic.

The full calculator is Decode String's park-and-resume applied to Basic Calculator II. Basic Calculator (224) and Basic Calculator III (772) ask for the value of an expression with parentheses: `"(1+(4+5+2)-3)+(6+8)"` is 23, and `"2*(5+5*2)/3"` is 10. A `(` parks `(terms so far, pending operator)`, and a finished group becomes one number for the outer expression. For 224 alone there is a shortcut: with only `+` and `-`, a parenthesis just flips signs, so a stack of signs is enough.

In [ ]:
def calculate(s):                            # + - * / and parentheses (224, 227, 772)
    stack, num, op, saved = [], 0, "+", []   # STATE + INIT: saved = (terms, op) parked at each "("
    for ch in s + "+":                       # the sentinel "+" flushes the last number
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch == "(":
            saved.append((stack, op))        # STEP: park the outer expression, start fresh
            stack, op = [], "+"
        elif ch in "+-*/)":
            push_term(stack, op, num)        # STEP: num is complete, apply the operator in front of it
            if ch == ")":                    # the group is finished: it becomes one number
                num = sum(stack)
                stack, op = saved.pop()      # resume the outer expression
            else:
                num, op = 0, ch
    return sum(stack)                        # RETURN


print(calculate("(1+(4+5+2)-3)+(6+8)"), calculate("-(2-3)"), calculate("2*(5+5*2)/3"))   # 23 1 10

**Try it**
- Print `saved` right after it grows, for `calculate("2*(3+(4-1))")`: first `[([2], '*')]`, then `[([2], '*'), ([3], '+')]`. Each `(` parks one more outer expression. The answer is 12.
- Delete `num = sum(stack)` and run `calculate("2*(3+4)")`: 8 instead of 14. The outer `*` multiplied only the group's last number.
- Predict `calculate("-(3*(2-5))")` before running: `-(3 * -3) = 9`.

### Say it in the interview

Warm-up (Valid Parentheses): "a closer must match the most recent unfinished opener, so I push openers, compare each closer with the top, and the string is valid iff the stack ends empty: O(n)."

The medium version, Decode String:

> "The brute force expands an innermost `k[...]` and rescans, again and again. But a `]` always closes the most recent open `[`, which is last-in-first-out, so one pass with a stack works. At `[` I park the text built so far and the count, and start fresh. At `]` I pop and glue prefix + inner × count. The invariant: `cur` is the innermost open group's text so far, and the stack holds the outer prefixes, innermost on top. Time is proportional to the output; the stack holds one frame per open bracket."

Point at the two pushes and pops while you say the invariant. Likely follow-ups and your answers:

- *Counts with several digits* → `k = k * 10 + int(ch)`, and reset `k` after `[`.
- *Text after a group*, `"2[a]b"` → a letter always extends `cur`, so it just works: `aab`.
- *Recursion instead of a stack* → each `[` calls the decoder for the inner part; the call stack holds the same frames.
- *A stray `]` with nothing open* → check `if not stack` before popping and report invalid input.
- *A calculator instead* → `*` and `/` happen at once, on the top term; `+` and `-` wait for the final sum; `int(a / b)` because Python's `//` floors.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Asteroid Collision | `stack/asteroid_collision.py` · `practice/simple/basics/stacks/05_asteroid_collision.py` | survivors on a stack; a left-mover fights right-moving tops while it keeps winning |
| Basic Calculator | `stack/basic_calculator.py` | only + and −: a parenthesis just flips the signs inside it; keep a stack of context signs |
| Basic Calculator II | `stack/basic_calculator_ii.py` · `practice/simple/basics/stacks/04_basic_calculator_ii.py` | apply the previous operator when the next one arrives; push signed terms, `*` `/` edit the top, then sum |
| Decode String | `stack/decode_string.py` · `practice/simple/basics/stacks/03_decode_string.py` | on `[` park (text, count) and start fresh; on `]` pop and glue prefix + inner × count |
| Design Circular Deque | `queues/design_circular_deque.py` | ring buffer at both ends: insertFront steps head back `(head − 1) % k` before writing |
| Design Circular Queue | `queues/design_circular_queue.py` | fixed array + head + count; tail = (head + count) % k; count tells empty from full |
| Dota2 Senate | `queues/dota2_senate.py` | one queue of turn indices per party; the earlier front bans the other and rejoins as i + n |
| Evaluate Reverse Polish Notation | `stack/evaluate_reverse_polish_notation.py` · `practice/simple/15_evaluate_reverse_polish_notation.py` | numbers wait on a stack; an operator pops the right operand, then the left, and pushes the result |
| Implement Queue using Stacks | `queues/implement_queue_using_stacks.py` · `practice/simple/basics/stacks/01_array_stack_and_queue_via_two_stacks.py` | inbox + outbox; pour only when the outbox is empty, so each item moves once: amortised O(1) |
| Implement Stack using Queues | `queues/implement_stack_using_queues.py` | after each push, rotate the queue len − 1 times so the newest item sits at the front |
| Longest Valid Parentheses | `stack/longest_valid_parentheses.py` | indices on a −1 barrier; after a match, run = i − top; a stray `)` becomes the barrier |
| Min Stack | `stack/min_stack.py` · `practice/simple/14_min_stack.py` | store (value, min so far) in every entry; popping reveals the older minimum for free |
| Minimum Add to Make Parentheses Valid | `stack/minimum_add_to_make_parentheses_valid.py` | one bracket type: the stack is a counter; a `)` at zero costs one insert, leftover opens one each |
| Number of Recent Calls | `queues/number_of_recent_calls.py` | t only grows, so expired pings sit at the front: popleft while q[0] < t − 3000 |
| Parsing a Boolean Expression | `stack/parsing_a_boolean_expression.py` | push everything; on `)` pop letters into a set down to `(`, then apply the operator below it |
| Valid Parentheses | `stack/valid_parentheses.py` · `practice/simple/13_valid_parentheses.py` | push openers; a closer must match the top; valid iff the stack ends empty |

### Self-check

1. Why can't Valid Parentheses be solved with one counter per bracket type?
<details><summary>Answer</summary>Counters forget the order. <code>"([)]"</code> has one of each and balanced counts, but its <code>)</code> arrives while <code>[</code> is the innermost open bracket. A counter is enough only with a single bracket type, because then every stack entry is the same and only the stack's height matters (921).</details>

2. In Basic Calculator II, why is an operator applied when the *next* operator arrives, not when it is read?
<details><summary>Answer</summary>When you read <code>*</code>, its right operand hasn't been read yet. The number is complete only when the next operator (or the sentinel at the end) shows up, so the pending operator waits in <code>op</code> and is applied then.</details>

3. Queue from two stacks: one `pop` can move n items. Why is it still O(1)?
<details><summary>Answer</summary>Amortised: every item is pushed onto the inbox once, moved to the outbox once and popped once, a constant number of stack operations per item. An unlucky single pop does many moves at once, but n operations never do more than a constant times n moves in total.</details>

4. The ring buffer keeps `head` and `count`. What goes wrong with only `head` and `tail`?
<details><summary>Answer</summary><code>head == tail</code> happens both when the queue is empty and when it is full, so you can't tell them apart. Keeping <code>count</code> (or always leaving one slot unused) removes the ambiguity, and the tail is just <code>(head + count) % k</code>.</details>

<a id="s08"></a>

## Monotonic Stack

> Keep a stack of the items that are still **waiting** for their answer. Each new item looks at the top: if it beats that item, the waiting item has just found its answer (pop it and record), and the newcomer looks at the next one down. Then the newcomer waits too. Beaten items leave, so the waiting values are always in sorted order: that is the "monotonic".

This section continues [Stacks & Queues](#s07) with one change. There, a closer or an operator popped the one entry it finished. Here a newcomer may finish several waiting items in a row, and the comparison that pops them keeps the waiting items sorted.

**Reach for it when** the problem asks, for every element, about the **nearest** element to its left or right that is **bigger or smaller**: the next warmer day, the next greater element, the previous smaller one, how far a bar can stretch, a stock's span. The same stack answers questions about the minimum or maximum of **every subarray**, and it builds the **smallest or largest sequence** when digits or letters may be deleted but their order must be kept.

**In this repo:** `stack/` (6 of its 16 problems: daily temperatures, next greater II, the two rectangle problems, car fleet, create maximum number) · bank: `practice/simple/16_daily_temperatures.py`, `practice/simple/17_largest_rectangle_in_histogram.py` · basics in `practice/simple/basics/monotonic_stacks/`: `01_next_greater_element.py`, `02_previous_smaller_element.py`, `03_online_stock_span.py`, `04_sum_of_subarray_minimums.py`, `05_remove_k_digits.py`

### The picture

```text
temps   73  74  75  71  69  72  76  73       question: how many days until a warmer day?
day      0   1   2   3   4   5   6   7

the days still waiting, just before day 5:   day 5 (72) arrives and looks at the top:

    #                                          69 < 72: pop day 4, it waited 5 - 4 = 1 day
    #   #                                      71 < 72: pop day 3, it waited 5 - 3 = 2 days
    #   #   #                                  75 > 72: stop, day 2 keeps waiting
   75  71  69                                  push day 5: the waiting temps are now 75 72
   d2  d3  d4
```

The waiting days always form a staircase that never steps up. A newcomer knocks down every step shorter than itself, and each knocked-down step has found its warmer day; then the newcomer becomes the new last step.

**Why it is fast:** the brute force scans forward from every day: O(n²). On a falling stretch like 75 71 69, the scan from 75 walks over 71 and 69, which are *themselves still waiting* for a warmer day, so neither can be 75's answer. The stack flips the question: instead of each day searching forward for its answer, each new day announces itself as the answer to every waiting day it beats. Every index is pushed once and popped at most once, so the whole pass is O(n), even with a `while` inside the `for`.

**Why it is correct:** when day `i` pops day `j`, `j` was still waiting, so no day between them was warmer than `j` (it would have popped `j` first). So `i` is the *first* warmer day after `j`, which is exactly the answer. A day that is never popped never met a warmer day.

### From idea to code

**The idea in one sentence:** *for each new item: while it beats the item on top of the stack, that item is resolved (pop it, record its answer); then push the new item, which now waits for its own answer.*

The **State** is a stack of **indices** of the items still waiting for their answer, and its **Definition** goes in a comment: `stack` holds the days whose warmer day has not come yet, temperatures never rising from bottom to top. It holds indices, not values, because the answer is written into `ans[j]` and is often a distance, `i - j`.

The **Invariant** has three parts. Every index left of `i` that is *not* on the stack already has its answer; the values on the stack are sorted; and every day between a waiting day `j` and today was no warmer than `j`, or it would have popped `j`. A **Step** pushes the newcomer, `stack.append(i)`, so that it waits for its own answer.

Before the step comes the **Fix**, `while stack and temps[stack[-1]] < t:`, which pops every waiting item the newcomer beats: a *next* question resolves them, a *previous* question discards them. **Record** at the pop for a *next* question, `ans[j] = i - j`, or `ans[j] = t` when the value is wanted, because the popped index has just met its answer. For a *previous* question, record after the fix and before the step, because the survivor on top is the answer for `i`.

**Init** is `ans = [0] * n` or `[-1] * n`, whichever means "never", and `stack = []`; sometimes a sentinel item at the end, such as `heights + [0]`, flushes whatever still waits. **Return** `ans`, where the indices never popped keep the default, or the best value recorded at the pops.

Which comparison pops is derived from the question instead of memorised. A *next* question pops while the newcomer *is* the top's answer, with the problem's own comparison: "strictly greater" pops while `top < x`, "greater or equal" while `top <= x`. A *previous* question pops while the top *cannot be* the newcomer's answer, which is the negation: for "strictly smaller", pop while `top >= x`, then read the top.

So next greater pops while the top is `< x`, and the waiting values never rise from bottom to top; next smaller pops while `> x`, and they never fall. Previous greater pops while `<= x` and then reads the top, on a stack that strictly falls; previous smaller pops while `>= x`, on a stack that strictly rises.

Each pop gives two facts at once: the popped item's next answer is the newcomer, and after the loop the top is the newcomer's previous answer, with the opposite strictness. Largest Rectangle, in Watch it work, uses both.

Two templates follow, one per kind of question. Daily Temperatures (739) asks, for each day, how many days until a warmer one, 0 if none ever comes: `[73, 74, 75, 71, 69, 72, 76, 73] → [1, 1, 4, 2, 1, 1, 0, 0]`. It is a *next* question, so the answer is written at the pop. FIX runs before STEP: pushed first, the newcomer would sit on top and block every comparison ([From Idea to Code](01_Start_Here.ipynb#s01) runs this swap).

Previous Smaller Element asks, for each item, for the nearest smaller value on its left, -1 if there is none: `[3, 1, 4, 1, 5] → [-1, -1, 1, -1, 1]`. It is a *previous* question, so the pops only discard: an item at least as big as `x` can never answer anyone again, because `x` is nearer and no bigger. After the loop the answer is `nums[stack[-1]] if stack else -1`, and then `x` waits.

In [ ]:
def daily_temperatures(temps):
    ans = [0] * len(temps)                   # INIT: 0 = "no warmer day ever comes"
    stack = []                               # STATE: days still waiting; temps never rise bottom -> top
    for i, t in enumerate(temps):
        while stack and temps[stack[-1]] < t:    # FIX: today beats the latest waiting day
            j = stack.pop()
            ans[j] = i - j                   # RECORD: j is resolved; i is its FIRST warmer day
        stack.append(i)                      # STEP: today waits for its own warmer day
    return ans                               # RETURN: never resolved -> stays 0


def previous_smaller(nums):
    ans, stack = [], []                      # STATE + INIT: indices; values strictly rise bottom -> top
    for i, x in enumerate(nums):
        while stack and nums[stack[-1]] >= x:    # FIX: not smaller than x, and x is nearer: useless now
            stack.pop()
        ans.append(nums[stack[-1]] if stack else -1)   # RECORD: the survivor on top is the answer
        stack.append(i)                      # STEP: x may answer later items
    return ans                               # RETURN


print(daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]))   # [1, 1, 4, 2, 1, 1, 0, 0]
print(previous_smaller([3, 1, 4, 1, 5]))                      # [-1, -1, 1, -1, 1]

**Try it**
- Use the rule to turn `previous_smaller` into *previous greater* (pop while `nums[stack[-1]] <= x`) and predict `[3, 1, 4, 1, 5]` before running: `[-1, 3, -1, 4, -1]`.
- Push the temperature instead of the index (`stack.append(t)`) and run the first example: `IndexError`, because the stack's entries are used as positions.
- Print `stack` right before the `return` for the first example: `[6, 7]`, the two days that never got a warmer day.
- In `previous_smaller`, change `>=` to `>` and run `previous_smaller([2, 2])`: `[-1, 2]` instead of `[-1, -1]`. An equal value is not smaller.

### Watch it work

The richest use of the loop is Largest Rectangle in Histogram (84). It asks for the largest rectangle inside a bar chart of width-1 bars: `[2, 1, 5, 6, 2, 3]` gives 10, height 5 across the bars of heights 5 and 6. The best rectangle of one bar has that bar's height and stretches until the first shorter bar on each side, so each bar needs its *previous smaller* and its *next smaller* bar, and one pop delivers both.

Bars pop when a bar that is not taller arrives, `>=`, so the newcomer is the popped bar's right wall. The bar under it on the stack is its left wall: its previous smaller bar, or -1, an imaginary wall before the array. The rectangle spans the bars strictly between the walls, and a final bar of height 0 pops whatever still waits. The trace prints every pop, then the heights still waiting after each step.

In [ ]:
def trace_rectangle(heights):
    bars, stack, best = heights + [0], [], 0
    for i, h in enumerate(bars):
        while stack and bars[stack[-1]] >= h:
            j = stack.pop()
            left = stack[-1] if stack else -1
            area = bars[j] * (i - left - 1)
            best = max(best, area)
            print(f"  bar {i} (h={h}) pops bar {j} (h={bars[j]}): walls {left} and {i}, width {i - left - 1}, area {area}")
        stack.append(i)
        print(f"i={i} h={h}  stack heights: {[bars[k] for k in stack]}")
    return best


print(trace_rectangle([2, 1, 5, 6, 2, 3]))   # 10

**Try it**
- Find the line that gives 10: bar 4 pops bar 2 (height 5), walls 1 and 4, width 2.
- Run `trace_rectangle([3, 3, 3])`: equal bars pop each other with widths 1 and 2 (under-measured), and only the last one gets width 3 (area 9). Harmless, which is why `>` and `>=` give the same answer.
- Run `trace_rectangle([1, 2, 3])`: nothing pops until the height-0 bar arrives, then all three pop at `i = 3`, and the best is 4.

### Where it goes wrong

1. **Storing values instead of indices.** You lose the position, so there is no way to write `ans[j]` or compute `i - j`: pushing temperatures makes `daily_temperatures([73, 74])` read `temps[73]`, an `IndexError`. Push `i`; read the value as `nums[stack[-1]]`.
2. **`<` vs `<=` when popping.** "Strictly warmer" pops on `<`. Popping on `<=` lets an equal temperature answer: `[70, 70, 71]` would give `[1, 1, 0]` instead of `[2, 1, 0]`.
3. **Recording in the wrong place.** *Next* answers are written for the **popped** index, at the pop. *Previous* answers are written for the **current** index, after the loop. Mixing them up gives answers to the wrong items: writing `ans[i] = i - j` at the pop turns `[73, 74]` into `[0, 1]` instead of `[1, 0]`.
4. **Forgetting the leftovers.** Items still on the stack at the end were never resolved. They need the default (`-1`, `0`), or, in the histogram, a final bar of height 0 so they get measured: `[1, 2, 3]` must give 4, but without the flush nothing is ever popped and you get 0.
5. **Width off by one.** When bar `j` pops at `i` and `left` is the index under it, the bars strictly between the walls number `i - left - 1`, not `i - left`. With `i - left`, `[2, 1, 5, 6, 2, 3]` gives 15 instead of 10.
6. **Counting ties twice.** In Sum of Subarray Minimums, which adds up the minimum of every subarray, `[2, 2]` has two equal minimums, and the subarray `[2, 2]` must be credited to exactly one of them: one wall strict, the other not. A single stack pass does this by itself; computing the left and right walls in two separate passes with the same comparison gives 8 or 4 instead of 6.
7. **Leftover deletions.** When you delete k digits to make the smallest number, digits that never fall (`"12345"`, k = 2) pop nothing, so cut the last k digits at the end. Then strip leading zeros, and return `"0"` if nothing is left (`"10"`, k = 2).

### Edge cases to say out loud

Empty input · one item · all equal (nothing is strictly greater) · strictly rising (everything resolves at once) · strictly falling (nothing resolves; everything is left at the end) · ties. And one habit worth more than any list: check the fast version against a brute force on many small random inputs.

In [ ]:
assert daily_temperatures([]) == []
assert daily_temperatures([50]) == [0]
assert daily_temperatures([70, 70, 70]) == [0, 0, 0]        # equal is not warmer
assert daily_temperatures([30, 40, 50]) == [1, 1, 0]        # each one resolved at once
assert daily_temperatures([50, 40, 30]) == [0, 0, 0]        # nobody is ever resolved
assert previous_smaller([]) == []
assert previous_smaller([2, 2]) == [-1, -1]                 # equal is not smaller
assert previous_smaller([1, 2, 3]) == [-1, 1, 2]


def brute_daily(temps):                      # O(n²) reference: scan forward from every day
    return [next((j - i for j in range(i + 1, len(temps)) if temps[j] > temps[i]), 0)
            for i in range(len(temps))]


random.seed(0)
for _ in range(300):                         # small random inputs, full of ties
    T = [random.randint(30, 35) for _ in range(random.randint(0, 9))]
    assert daily_temperatures(T) == brute_daily(T), T
print("edge cases pass")

**Try it**
- Predict `previous_smaller([5, 4, 3])` and `previous_smaller([1, 3, 2])` before running: `[-1, -1, -1]` and `[-1, 1, 1]`.
- Break `daily_temperatures` on purpose (pop on `<=`) and rerun this cell: the `[70, 70, 70]` assert fails first. Delete that assert and rerun: the random check fails too, and its message shows the input it failed on.
- Add `assert daily_temperatures([50] * 100_000) == [0] * 100_000`: it runs instantly. Every index is pushed once and never popped.

### Variations

| Variation | What changes from the template | Problems |
|---|---|---|
| **Next greater value** | record `ans[j] = x` instead of the distance | Next Greater Element I (496): the next greater value of each item of a subset, looked up in the full array |
| **Next greater in a linked list** | copy the values into a list, then run the template | Next Greater Node In Linked List (1019) |
| **Circular array** | walk `2n` steps with `i % n`; push only during the first lap | Next Greater Element II (503): the array wraps around |
| **Previous smaller / greater** | pop the useless items, then read the top for the current item | Largest Rectangle in Histogram (84); Online Stock Span (901): how many days in a row up to today had a price at most today's |
| **Span, online** | store `(price, span)`; a popped day's span is absorbed into today's | Online Stock Span (901) |
| **Sort first, then stack** | sort by position; a car whose solo arrival time is later than the fleet ahead's starts a new fleet | Car Fleet (853): how many groups of cars reach the target |
| **Greedy smallest / largest sequence** | pop while the top is worse than `x` *and* you can still afford to drop it | Remove K Digits (402): the smallest number after k deletions; Smallest Subsequence of Distinct Characters (1081): each letter once, smallest order |
| **Both walls at once** | rising stack: when `j` pops at `i`, `i` is its right wall and the new top its left wall | Largest Rectangle in Histogram (84); Trapping Rain Water (42), the water held between bars, has a stack version too, but is taught in [Two Pointers](03_Two_Pointers_Sliding_Window_Strings.ipynb#s05) |
| **Contribution counting** | `nums[j]` is the minimum of `(j - left) * (i - j)` subarrays | Sum of Subarray Minimums (907); Sum of Subarray Ranges (2104), the total of max − min over all subarrays, is 907 twice |
| **Both walls + prefix sums** | each minimum's walls bound the widest subarray it rules; its sum comes from prefix sums | Maximum Subarray Min-Product (1856): the best minimum × sum over all subarrays |
| **Scan from the right** | walk right to left; the last value popped is the best "2" | 132 Pattern (456): is there `i < j < k` with `nums[i] < nums[k] < nums[j]`? |
| **Candidates, then a second scan** | keep a falling stack of left candidates, then match them from the right | Maximum Width Ramp (962): the widest `i < j` with `nums[i] <= nums[j]` |
| **Monotonic deque** | this stack plus expiry from the front: the window maximum | Sliding Window Maximum (239), in [Sliding Window](03_Two_Pointers_Sliding_Window_Strings.ipynb#s06) |
| *Second pass:* **Rows into histograms** | each row of a binary matrix becomes a histogram of the 1s standing on it; run Largest Rectangle per row | Maximal Rectangle (85): the largest rectangle of 1s in a binary matrix |
| *Second pass:* **Largest sequence from two arrays** | keep the largest t digits of each array with the mirror-image stack; try every split of k; merge by comparing remaining tails | Create Maximum Number (321): the largest k-digit number from two digit arrays, each keeping its order |

The first variation changes only the input. Next Greater Element II (503) asks for the next greater value of every item in a circular array, where the first item follows the last, -1 if there is none: `[1, 2, 1] → [2, -1, 2]`. Walk the array twice: the second lap lets the items at the start answer the items still waiting at the end. Only the first lap pushes, so each index waits once.

In [ ]:
def next_greater_circular(nums):
    n = len(nums)
    ans, stack = [-1] * n, []                # STATE + INIT: indices still waiting; -1 = "never"
    for i in range(2 * n):                   # two laps around the circle
        x = nums[i % n]
        while stack and nums[stack[-1]] < x: # FIX: x answers every smaller waiting item
            ans[stack.pop()] = x             # RECORD
        if i < n:
            stack.append(i)                  # STEP: only the first lap creates waiting items
    return ans                               # RETURN


print(next_greater_circular([1, 2, 1]), next_greater_circular([1, 2, 3, 4, 3]))   # [2, -1, 2] [2, 3, 4, -1, 4]

**Try it**
- Delete `if i < n:` so the second lap pushes too: `IndexError`, since `i` (not `i % n`) is then used as a position past the end of `nums`.
- Walk one lap only (`range(n)`) and run `[1, 2, 1]`: `[2, -1, -1]`. The last 1 never sees the 2 at the front.
- Predict `[5, 4, 3, 2, 1]` and `[3, 3, 3]` before running: `[-1, 5, 5, 5, 5]` and `[-1, -1, -1]`.

The *previous* question comes next, asked online, so the stack lives between calls. Online Stock Span (901) gets one price per call and returns how many consecutive days, ending today, had a price at most today's: prices 100, 80, 60, 70, 60, 75, 85 give 1, 1, 1, 2, 1, 4, 6. A day that today beats lies inside today's span, and so does every day it had already absorbed, so the stack stores `(price, span)` pairs and today adds up the spans it pops.

Car Fleet (853) sits at the other extreme: a monotonic stack that never pops. Cars drive toward a target, a car that catches a slower one slows down and joins its fleet, and the question is how many fleets arrive: target 12, positions `[10, 8, 0, 5, 3]`, speeds `[2, 4, 1, 1, 3]` → 3. Sort by position and walk from the car closest to the target, comparing *solo arrival times*, not speeds.

A car whose time is later than the fleet ahead's can never catch it and leads a new fleet; one that would arrive at the same time or sooner catches up and merges. The times on the stack only rise, which is why nothing pops: the stack is really a running maximum.

In [ ]:
class StockSpanner:                          # 901
    def __init__(self):
        self.stack = []                      # STATE + INIT: (price, span); prices strictly fall bottom -> top

    def next(self, price):
        span = 1                             # today itself
        while self.stack and self.stack[-1][0] <= price:   # FIX: absorb every day today beats
            span += self.stack.pop()[1]      # ... and every day that day had covered
        self.stack.append((price, span))     # STEP
        return span                          # RECORD + RETURN


def car_fleet(target, position, speed):      # 853
    cars = sorted(zip(position, speed), reverse=True)   # INIT: closest to the target first
    times = []                               # STATE: arrival times of the fleets, front fleet first
    for pos, spd in cars:
        t = (target - pos) / spd             # this car's solo arrival time
        if not times or t > times[-1]:       # arrives later than the fleet ahead: a new fleet
            times.append(t)                  # STEP (otherwise it catches up and merges)
    return len(times)                        # RETURN


spanner = StockSpanner()
print([spanner.next(p) for p in [100, 80, 60, 70, 60, 75, 85]])   # [1, 1, 1, 2, 1, 4, 6]
print(car_fleet(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]))           # 3

**Try it**
- Change `<=` to `<` in `next` and replay `[60, 60, 60]` on a fresh spanner: `[1, 1, 1]` instead of `[1, 2, 3]`.
- Count each popped day as 1 (`span += 1`) instead of adding its span, and replay the main prices: `[1, 1, 1, 2, 1, 3, 3]`. The 75 loses the day that the 70 had absorbed, and the 85 gets 3 instead of 6.
- In `car_fleet`, drop `reverse=True` and rerun: 1 instead of 3. Each car now compares itself with a car *behind* it.
- Run `car_fleet(10, [0, 5], [2, 1])`: 1, because both cars reach the target at t = 5 and arrive together. With `t >= times[-1]` it would say 2.

The stack can also *build* an answer instead of answering questions. Remove K Digits (402) asks for the smallest number left after deleting k digits from a number string: `"1432219"`, k = 3 → `"1219"`. The leftmost digits matter most, so a digit sitting right before a smaller one should go, because deleting it slides the smaller digit into a more important place. The kept digits should therefore never fall: pop while the top is bigger than the newcomer and deletions are left.

Smallest Subsequence of Distinct Characters (1081), the same problem as Remove Duplicate Letters (316), keeps every distinct letter exactly once and wants the smallest such string: `"cbacdcbc"` → `"acdb"`. It swaps the budget for a different test: a bigger letter on top is dropped only if it appears again later.

In [ ]:
def remove_k_digits(num, k):
    stack = []                               # STATE + INIT: kept digits, never falling
    for d in num:
        while k and stack and stack[-1] > d: # FIX: a bigger digit before a smaller one: drop it
            stack.pop()
            k -= 1
        stack.append(d)                      # STEP
    stack = stack[:len(stack) - k]           # deletions left over come off the tail
    return "".join(stack).lstrip("0") or "0" # RETURN


def smallest_subsequence(s):                 # every distinct letter exactly once
    last = {c: i for i, c in enumerate(s)}   # the last position of each letter
    stack, used = [], set()
    for i, c in enumerate(s):
        if c in used:
            continue
        while stack and stack[-1] > c and last[stack[-1]] > i:   # FIX: it comes back later: drop it
            used.remove(stack.pop())
        stack.append(c)                      # STEP
        used.add(c)
    return "".join(stack)


print(remove_k_digits("1432219", 3), remove_k_digits("10200", 1), remove_k_digits("10", 2))   # 1219 200 0
print(smallest_subsequence("bcabc"), smallest_subsequence("cbacdcbc"))                      # abc acdb

**Try it**
- Delete the line that trims leftover deletions and run `remove_k_digits("12345", 2)`: `12345` instead of `123`.
- Drop `or "0"` and run `remove_k_digits("10", 2)`: an empty string instead of `0`.
- In `smallest_subsequence`, delete `and last[stack[-1]] > i` and run `"cbacdcbc"`: `abc`. The `d` was dropped although it never comes back.
- Print `stack` after every digit of `"1432219"` with k = 3: the 4, the 3 and one 2 are popped, each by the smaller digit right after it, and `['1', '2', '1', '9']` is left.

Both walls at once come next, in the loop traced in Watch it work, without the prints. Bar `j`'s best rectangle has its own height and stretches until the first shorter bar on each side, and the pop delivers both walls: `i` on the right, the new top on the left. `[2, 1, 5, 6, 2, 3]` gives 10, and `[2, 4]` gives 4, from the bar of height 4 alone or from both bars at height 2.

In [ ]:
def largest_rectangle(heights):
    bars = heights + [0]                     # a height-0 bar resolves everything at the end
    stack, best = [], 0                      # STATE + INIT: indices; heights rise bottom -> top
    for i, h in enumerate(bars):
        while stack and bars[stack[-1]] >= h:    # FIX: the top can't stretch past i
            height = bars[stack.pop()]
            left = stack[-1] if stack else -1    # its left wall: the nearest shorter bar
            best = max(best, height * (i - left - 1))   # RECORD: bars strictly between the walls
        stack.append(i)                      # STEP
    return best                              # RETURN


print(largest_rectangle([2, 1, 5, 6, 2, 3]), largest_rectangle([2, 4]))   # 10 4

**Try it**
- Drop the sentinel (`bars = heights`) and run `largest_rectangle([1, 2, 3])`: 0 instead of 4. The heights only rose, so no bar was ever popped and measured.
- Change the width to `i - left` and run `[2, 1, 5, 6, 2, 3]` again: 15 instead of 10. Every rectangle now counts one bar too many (height 5 gets width 3).
- Predict `largest_rectangle([6, 2, 5, 4, 5, 1, 6])` before running: 12, height 4 across the three bars 5, 4 and 5. Print `(height, i - left - 1)` at each pop to see which pop finds it.

Counting with both walls closes the main path. Sum of Subarray Minimums (907) asks for the sum of `min(sub)` over every contiguous subarray, modulo 10⁹ + 7: `[3, 1, 2, 4]` → 17. Turn the sum around: instead of finding the minimum of every subarray, ask for every element *how many subarrays it is the minimum of*.

With its walls at `left` and `i`, the nearest smaller items on each side, `nums[j]` is the minimum of every subarray that starts in `left+1 .. j` and ends in `j .. i-1`: that is `(j - left) * (i - j)` subarrays. The histogram's pop hands you both walls, and making one wall strict and the other not counts a tie exactly once.

In [ ]:
def sum_subarray_mins(nums):
    vals = nums + [-math.inf]                # sentinel: smaller than everything, pops all
    stack, total = [], 0                     # STATE + INIT: indices; values rise bottom -> top
    for i, x in enumerate(vals):
        while stack and vals[stack[-1]] >= x:    # FIX
            j = stack.pop()                  # i = right wall: the first value <= vals[j]
            left = stack[-1] if stack else -1    # left wall: the last value < vals[j]
            total += vals[j] * (j - left) * (i - j)   # RECORD: vals[j] is the min of that many subarrays
        stack.append(i)                      # STEP
    return total % (10**9 + 7)               # RETURN


print(sum_subarray_mins([3, 1, 2, 4]), sum_subarray_mins([11, 81, 94, 43, 3]), sum_subarray_mins([2, 2]))   # 17 444 6

**Try it**
- Check it against the brute force: with `nums = [3, 1, 2, 4]`, `sum(min(nums[i:j]) for i in range(4) for j in range(i + 1, 5))` is also 17.
- Print `(vals[j], j - left, i - j)` at each pop for `[3, 1, 2, 4]`: the 1 is the minimum of 2 × 3 = 6 subarrays.
- Change `>=` to `>`: still 17, and still 6 for `[2, 2]`. One pass always makes one wall strict and the other not, so a tie is counted once either way.

The rest of this section is a second pass: Hard problems that reuse the same moves. Skip them until the main path is automatic.

Maximal Rectangle (85) asks for the largest rectangle made only of 1s in a binary matrix; in the 4 × 5 grid of the cell, the answer is 6, two rows of three 1s. Turn each row into a histogram: `heights[c]` counts the 1s standing on this row in column c, and a `"0"` resets its column to the ground. The largest rectangle that sits on this row is then `largest_rectangle(heights)`, and the answer is the best over all rows.

In [ ]:
def maximal_rectangle(matrix):
    best, heights = 0, [0] * (len(matrix[0]) if matrix else 0)
    for row in matrix:
        for c, cell in enumerate(row):       # the 1s standing on this row, per column
            heights[c] = heights[c] + 1 if cell == "1" else 0
        best = max(best, largest_rectangle(heights))
    return best


grid = [["1", "0", "1", "0", "0"], ["1", "0", "1", "1", "1"],
        ["1", "1", "1", "1", "1"], ["1", "0", "0", "1", "0"]]
print(maximal_rectangle(grid), maximal_rectangle([["0"]]), maximal_rectangle([]))   # 6 0 0

**Try it**
- Print `heights` after each row in `maximal_rectangle(grid)`: `[1, 0, 1, 0, 0]`, `[2, 0, 2, 1, 1]`, `[3, 1, 3, 2, 2]`, `[4, 0, 0, 3, 0]`. A `"0"` resets its column to the ground.
- Print `largest_rectangle(heights)` after each row as well: 1, 3, 6 and 4. The best rectangle sits on the third row.
- Compare with the number instead of the character (`cell == 1`) and rerun: 0. The matrix holds the characters `"0"` and `"1"`, and no character equals the number 1.

Create Maximum Number (321) asks for the largest number of k digits taken from two digit arrays, each keeping its own order: `[3, 4, 6, 5]` and `[9, 1, 2, 5, 8, 3]` with k = 5 give `[9, 8, 6, 5, 3]`. The full solution tries every split of k between the two arrays, takes the best subsequence of each, and merges the two by comparing the remaining tails, not just the heads.

Its building block is Remove K Digits mirrored. To keep the *largest* t digits of one array in order, pop while the top is *smaller* than the newcomer and drops are left; drops that are left over come off the tail, which `stack[:t]` does.

In [ ]:
def max_subsequence(nums, t):                # 321's building block: the largest t digits, in order
    drop, stack = len(nums) - t, []
    for x in nums:
        while drop and stack and stack[-1] < x:   # FIX: the mirror image, drop smaller digits
            stack.pop()
            drop -= 1
        stack.append(x)                      # STEP
    return stack[:t]


print(max_subsequence([9, 1, 2, 5, 8, 3], 3), max_subsequence([3, 4, 6, 5], 2))   # [9, 8, 3] [6, 5]

**Try it**
- Predict `max_subsequence([6, 0, 4], 2)` before running: `[6, 4]`.
- Delete the `[:t]` and run `max_subsequence([5, 4, 3], 1)`: `[5, 4, 3]` instead of `[5]`. Falling digits never pop, so the unused drops must come off the tail, as in Remove K Digits.
- Change `<` to `<=` and run `max_subsequence([6, 6, 5], 2)`: `[6, 5]` instead of `[6, 6]`. Popping an equal digit spends a drop for nothing.
- Finish 321: for every split `i + (k - i) = k` that both arrays can supply, merge `a = max_subsequence(nums1, i)` and `b = max_subsequence(nums2, k - i)` with `[max(a, b).pop(0) for _ in a + b]`, which compares the remaining tails, and keep the largest result. The example gives `[9, 8, 6, 5, 3]`.

### Say it in the interview

> "The brute force scans right from every item for the first bigger one: O(n²). Those scans keep walking over items that are themselves still waiting. So I'll flip it: a stack holds the indices still waiting, whose values never rise; each new item pops and answers every smaller item on top, then waits itself. When `i` pops `j`, `j` was still waiting, so nothing between them beat it: `i` is `j`'s first warmer day. Every index is pushed once and popped once, so it's O(n) time and O(n) space."

Then point at the pop line and say *who* is resolved there ("this index just met its next warmer day"), and say what is left on the stack at the end and which answer those items get. Likely follow-ups and your answers:

- *Values instead of distances* (496) → record `ans[j] = t`.
- *A circular array* (503) → walk two laps with `i % n`; push only during the first.
- *The previous item instead of the next* → record after the loop, before pushing.
- *A stream of prices* (901) → the same stack, kept between calls; each call is amortised O(1), which means O(1) on average over all calls, because each price is pushed once and popped at most once.
- *O(1) extra space for 739* → scan right to left; to find day `i`'s answer, start at `j = i + 1` and jump `j += ans[j]` over days that are not warmer.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Car Fleet | `stack/car_fleet.py` | sort by position, closest first; compare solo arrival times: later than the fleet ahead → new fleet, same or sooner → merges |
| Create Maximum Number | `stack/create_maximum_number.py` | try every split; best subsequence per array by monotonic stack; merge by comparing remaining tails, not heads |
| Daily Temperatures | `stack/daily_temperatures.py` · `practice/simple/16_daily_temperatures.py` | stack of days still waiting (temperatures never rising); a warmer day pops them and records i − j |
| Largest Rectangle in Histogram | `stack/largest_rectangle_in_histogram.py` · `practice/simple/17_largest_rectangle_in_histogram.py` | rising stack; a popped bar's walls are the current index and the new top; flush with a 0 bar |
| Maximal Rectangle | `stack/maximal_rectangle.py` | each row is a histogram of the 1s standing on it; run Largest Rectangle once per row |
| Next Greater Element | `practice/simple/basics/monotonic_stacks/01_next_greater_element.py` | waiting indices; a bigger value pops and answers them; leftovers keep −1 |
| Next Greater Element II | `stack/next_greater_element_ii.py` | circular: walk 2n steps with i % n and push only during the first lap |
| Online Stock Span | `practice/simple/basics/monotonic_stacks/03_online_stock_span.py` | (price, span) pairs; a price ≤ today's is absorbed with every day it covered |
| Previous Smaller Element | `practice/simple/basics/monotonic_stacks/02_previous_smaller_element.py` | pop everything ≥ x (x is nearer and no bigger), then the top is the answer |
| Remove K Digits | `practice/simple/basics/monotonic_stacks/05_remove_k_digits.py` | drop a digit bigger than the next while k lasts; trim the tail; strip zeros; "0" if empty |
| Sum of Subarray Minimums | `practice/simple/basics/monotonic_stacks/04_sum_of_subarray_minimums.py` | nums[j] is the min of (j − left)(i − j) subarrays; one wall strict, the other not |

### Self-check

1. Why does the stack hold indices rather than values?
<details><summary>Answer</summary>The answer is written for a specific position (<code>ans[j]</code>) and often needs a distance or a width (<code>i - j</code>, <code>i - left - 1</code>). A value loses its position, while an index can always give the value back as <code>nums[j]</code>.</details>

2. In Daily Temperatures, when exactly is a day resolved, and what happens to the days still on the stack at the end?
<details><summary>Answer</summary>A day is resolved at the moment it is popped: the current day is the first one warmer than it, so its answer is <code>i - j</code>. Days still waiting at the end never met a warmer day, so they keep the default 0.</details>

3. The loop has a `while` inside a `for`. Why is it O(n) and not O(n²)?
<details><summary>Answer</summary>Count pops instead of loops. Each index is pushed exactly once, so it can be popped at most once, and all the <code>while</code> loops together pop at most n times. The total work is O(n).</details>

4. In Largest Rectangle, why is the left wall of a popped bar simply the new top of the stack?
<details><summary>Answer</summary>It is <code>previous_smaller</code> in disguise. When bar j arrived, it popped every bar at least as tall, so the bar left under it is its previous shorter bar, and nothing under an entry changes while that entry waits.</details>

<a id="s10"></a>

## Linked Lists

> A linked list is a chain of boxes, each holding a value and **one** arrow to the next box. You can't index into it, walk it backwards or see its length; all you can do is put named fingers on boxes (`prev`, `cur`, `slow`, `fast`) and re-aim arrows. Every algorithm is careful pointer surgery: **give every box you will touch a name, then rewire.**

[Stacks & Queues](#s07) and [Monotonic Stack](#s08) kept their items in a Python list, where the order is given and any index is one step away. Here the order itself is made of arrows, so changing it safely is the whole skill: name every box, rewire, move the fingers last.

**Reach for it when** the input is a `ListNode`; the problem says *in place* or *O(1) extra space*; it asks for the **middle**, the **k-th from the end**, a **cycle**, a **merge** or a **reversal**; or an array's values are indices into the same array, because then the array *is* a linked list and Floyd's cycle trick applies.

**In this repo:** `linked_list/` (12 problems) · bank: `practice/simple/21_reverse_linked_list.py`, `practice/simple/22_merge_two_sorted_lists.py`, `practice/simple/23_linked_list_cycle_ii.py`, `practice/simple/25_remove_nth_node_from_end.py` · basics: `practice/simple/basics/linked_lists/` (build/insert/delete, reverse iterative and recursive, middle, Floyd, merge, palindrome, k-group, add two numbers). LRU Cache (146), a fixed-size cache that evicts the least recently used key, is a dict plus a doubly linked list and lives in [Design Problems](11_Design.ipynb#s24).

### The picture

```text
A box holds a value and ONE arrow. A variable is a finger pointing at a box.

  head
   v
  [1]->[2]->[3]->None

Delete 2: put a finger on the box BEFORE it and re-aim that box's arrow past the victim.

  prev
   v
  [1]  [2]->[3]->None          prev.next = prev.next.next
   |         ^                 (nothing points at 2 any more: it is gone)
   +---------+

A dummy box in front gives even the head a box before it:

  [D]->[1]->[2]->None          deleting the head is just dummy.next = dummy.next.next,
                               and the answer is always dummy.next
```

Most steps change more than one arrow. Make it mechanical: draw 3 or 4 boxes before and after **one** step, then type three moves.

```text
  before:  [prev]->[a]->[b]->[after]          after:  [prev]->[b]->[a]->[after]

  1. NAME    every box the step touches:     a, b, after = prev.next, prev.next.next, prev.next.next.next
  2. REWIRE  the changed arrows, any order:  prev.next, b.next, a.next = b, a, after
  3. MOVE    the fingers last:               prev = a

Once every box has a name, no write can lose anything, so the order of the writes stops mattering.
Inserting x after prev is the same recipe: name after = prev.next, then prev.next, x.next = x, after.
```

The fingers can also *measure*, which is how a list answers questions an array answers with an index:

```text
middle: slow moves 1 box, fast moves 2; when fast can't take two more steps, slow is halfway

  [1]->[2]->[3]->[4]->[5]->None
             ^         ^
           slow      fast

n-th from the end (n = 2): open a gap of n + 1 links, then slide both until fast falls off

  [D]->[1]->[2]->[3]->[4]->[5]->None
                  ^               ^
                slow            fast          slow.next (4) is 2nd from the end: skip it
```

**Why it is fast and small:** the brute force for almost every list problem is "copy the nodes into a Python list, use indices, rebuild". That is O(n) time but O(n) extra space, and the interviewer will ask you to drop it.

Each list technique replaces one array superpower with a pointer trick. *Index from the end* becomes a finger trailing by a fixed gap; *the middle* becomes a finger at half speed; *walk backwards* becomes "reverse that half in place"; *have I been here before?* becomes a fast finger lapping a slow one. Fingers only move forward, so each trick is O(n) time and O(1) space.

### From idea to code

**The idea in one sentence:** *name every box a step touches, rewire the arrows, move the fingers last, and put a dummy box in front so the head is never a special case.*

The **State** is a few fingers: `prev` and `cur` to rewire, `tail` to build a new chain, `slow` and `fast` to measure, and often a `dummy` box in front of the head. Their **Definition** goes in a comment: `prev` is the head of the already-reversed part, `cur` the first box not processed yet, `tail` the last box of the answer so far. The **Invariant**: everything before `cur` is done and reachable from `prev` or `dummy`, and everything from `cur` on is untouched and reachable from `cur`.

A **Step** names every box it touches, `nxt = cur.next`, rewires the arrows, `cur.next = prev`, and only then moves the fingers, `prev, cur = cur, nxt`. Where to **Record** depends on the job. Building, you hang a box on the end of the answer, `tail.next = node`, then `tail = tail.next`. Rewiring in place records nothing, because the structure *is* the answer. Finding records the moment the fingers meet or stop, and then acts on that box: return it, or skip it with `prev.next = prev.next.next`.

**Init** is `dummy = ListNode(0, head)`, or `prev, cur = None, head` for a reversal, or `slow = fast = head`, sometimes the dummy, for measuring. **Return** `dummy.next`, never `head`, because the head may have moved or been deleted; `prev` after a reversal; a box or `None` for a search. Two phrases translate the same way everywhere: "can fast take two more steps?" is `while fast and fast.next:`, and "the same box", not the same value, is `a is b`.

Every cell below needs the box itself and two helpers. `ListNode` holds a value and one arrow. `build_list([1, 2, 3])` hangs one box per value behind a dummy and returns the head, and `to_list` walks the arrows back into a Python list so that a result can be printed; its `limit` stops a walk that would go round a cycle forever.

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next


def build_list(values):                      # [1, 2, 3] -> 1 -> 2 -> 3, returns the head
    dummy = tail = ListNode()
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head, limit=1000):               # 1 -> 2 -> 3 -> [1, 2, 3]
    out = []
    while head and len(out) < limit:         # the limit stops a walk around a cycle
        out.append(head.val)
        head = head.next
    return out


head = build_list([1, 2, 3])
print(head.val, head.next.val, head.next.next.next)   # 1 2 None
print(to_list(head))                                  # [1, 2, 3]

**Try it**
- Run `build_list([])` and `to_list(None)`: `None` and `[]`. An empty list is just "no first box", the first edge case to say out loud.
- Make a cycle by hand: `h = build_list([1, 2, 3]); h.next.next.next = h`, then `to_list(h, 7)` gives `[1, 2, 3, 1, 2, 3, 1]`. Without the limit this walk would never end: remember that when a bug creates a cycle.
- Build `[1, 2, 3]` and print `head.next.next.next is None`: `True`. The last box's arrow points at nothing, which is what every `while cur:` loop waits for.

Three templates cover most problems: rewire in place, build a new chain, and fingers that measure; in each, the order of the tagged lines is itself a decision. Reverse Linked List (206) rewires in place: `1 → 2 → 3 → 4 → 5` becomes `5 → 4 → 3 → 2 → 1`. It names `nxt` before the flip, because flipping first loses the rest of the list, and it moves the fingers last.

Merge Two Sorted Lists (21) builds one sorted chain from the boxes of two: `[1, 2, 4]` and `[1, 3, 4]` give `[1, 1, 2, 3, 4, 4]`. It hangs the smaller front box on the answer (RECORD) before `tail` steps onto it (STEP), and when one list runs out, the rest of the other is hung on whole.

The last two measure. Middle of the Linked List (876) returns the middle box, the second of the two middles on an even length: 3 for both `[1, 2, 3, 4, 5]` and `[1, 2, 3, 4]`. Remove Nth Node From End of List (19) deletes the n-th box from the end, `[1, 2, 3, 4, 5]` with n = 2 → `[1, 2, 3, 5]`, and acts only after the slide, when `slow` is known to sit just before the victim.

In [ ]:
def reverse_list(head):
    prev, cur = None, head                   # STATE + INIT: prev = reversed part, cur = the rest
    while cur:
        nxt = cur.next                       # 1. name the box you would lose
        cur.next = prev                      # 2. STEP: flip one arrow
        prev, cur = cur, nxt                 # 3. move both fingers one box right
    return prev                              # RETURN: the old last box is the new head


def merge_lists(a, b):
    dummy = tail = ListNode()                # STATE + INIT: tail = last box of the answer
    while a and b:
        if a.val <= b.val:                   # the smaller front box goes next
            tail.next, a = a, a.next         # RECORD: hang it on the answer
        else:
            tail.next, b = b, b.next
        tail = tail.next                     # STEP: tail moves onto the box just hung
    tail.next = a or b                       # the leftover is already sorted: hang it whole
    return dummy.next                        # RETURN: skip the dummy


def middle_node(head):
    slow = fast = head                       # STATE + INIT: both fingers on the head
    while fast and fast.next:                # while fast can take two more steps
        slow, fast = slow.next, fast.next.next   # STEP: slow moves 1 box, fast moves 2
    return slow                              # RETURN: the middle (the 2nd middle if even)


def remove_nth_from_end(head, n):
    dummy = ListNode(0, head)                # INIT: a box before the head, so the head can go too
    slow = fast = dummy                      # STATE: two fingers that will stay n + 1 links apart
    for _ in range(n + 1):
        fast = fast.next                     # INIT: open the gap of n + 1 links
    while fast:                              # until fast falls off the end
        slow, fast = slow.next, fast.next    # STEP: both move one box; the gap stays n + 1
    slow.next = slow.next.next               # RECORD: slow stopped just before the victim; skip it
    return dummy.next                        # RETURN: never `head` (it may be gone)


print(to_list(reverse_list(build_list([1, 2, 3, 4, 5]))))                   # [5, 4, 3, 2, 1]
print(to_list(merge_lists(build_list([1, 2, 4]), build_list([1, 3, 4]))))   # [1, 1, 2, 3, 4, 4]
print(middle_node(build_list([1, 2, 3, 4, 5])).val, middle_node(build_list([1, 2, 3, 4])).val)   # 3 3
print(to_list(remove_nth_from_end(build_list([1, 2, 3, 4, 5]), 2)))         # [1, 2, 3, 5]

**Try it**
- Swap the first two lines inside `reverse_list`'s loop (flip, *then* name) and reverse `[1, 2, 3, 4, 5]`: you get `[1]`. After the flip, `cur.next` is `None`, so `nxt` named nothing and the rest of the list is lost.
- In `remove_nth_from_end`, change `range(n + 1)` to `range(n)`: `[1, 2, 3, 4, 5], 2` now gives `[1, 2, 3, 4]`. With a gap of n, `slow` stops *on* the 4 instead of before it, so the wrong box is skipped.
- Change `while fast and fast.next:` to `while fast.next:` and call `middle_node(build_list([1, 2, 3, 4]))`: `AttributeError`, because on an even length `fast` itself becomes `None`. Odd lengths still work, which is why this bug survives a quick test.
- Delete `tail.next = a or b` from `merge_lists`: the merge prints `[1, 1, 2, 3, 4]`. The second list's `4` was still waiting when the first list ran out, so it was never hung on the answer.

Two everyday rules come next, each with its own problem. Swap Nodes in Pairs (24) swaps every two neighbouring boxes, `[1, 2, 3, 4, 5] → [2, 1, 4, 3, 5]`, and it is the three-move recipe from the picture, word for word. Remove Linked List Elements (203) deletes every box that holds a given value, `[1, 2, 6, 3, 6, 6]` without its 6s → `[1, 2, 3]`, and it adds the other rule: after a delete, don't move. The finger must look at the *new* `cur.next`, which nobody has checked yet.

In [ ]:
def swap_pairs(head):
    dummy = ListNode(0, head)
    prev = dummy                                  # STATE: the box before the next pair
    while prev.next and prev.next.next:
        a, b, after = prev.next, prev.next.next, prev.next.next.next   # 1. name every box you touch
        prev.next, b.next, a.next = b, a, after   # 2. STEP: every box has a name, so any order works
        prev = a                                  # 3. move last: a is now the box before the next pair
    return dummy.next                             # RETURN


def remove_elements(head, val):
    dummy = ListNode(0, head)
    cur = dummy                                   # STATE: the last box we keep
    while cur.next:
        if cur.next.val == val:
            cur.next = cur.next.next              # STEP: delete, and stay: check the new cur.next
        else:
            cur = cur.next                        # keep it, then move on
    return dummy.next                             # RETURN


print(to_list(swap_pairs(build_list([1, 2, 3, 4, 5]))))           # [2, 1, 4, 3, 5]
print(to_list(remove_elements(build_list([1, 2, 6, 3, 6, 6]), 6)))  # [1, 2, 3]

**Try it**
- Write the arrows in another order, `a.next, prev.next, b.next = after, b, a`: still `[2, 1, 4, 3, 5]`. With every box named, the order of the writes doesn't matter.
- Now leave `after` unnamed: name only `a, b`, and write `prev.next = b`, then `b.next = a`, then `a.next = b.next` on three lines. First add a step guard (`steps = 0` before the loop; `steps += 1` and `if steps > 20: break` inside), because the loop never ends: on `[1, 2, 3, 4]`, `to_list(swap_pairs(...), 6)` shows `[2, 1, 1, 1, 1, 1]`. `b.next = a` erased the only arrow to 3, so `a.next = b.next` made `a` point at itself.
- In `remove_elements`, move the finger after every step (replace the `if/else` with the delete under an `if`, then an unconditional `cur = cur.next`): removing 2 from `[1, 2, 2, 3]` gives `[1, 2, 3]`, and removing 6 from `[6, 6, 1]` gives `[6, 1]`. The second copy was never checked.
- Remove every value from `[7, 7, 7]`: `[]`. The dummy is the only box left, which is why the loop watches `cur.next` and not `cur`.

### Watch it work

The reversal keeps two chains: the reversed part, drawn with its arrows pointing left and `prev` as its rightmost box, and the untouched rest, which starts at `cur`. Each step moves one box from the right chain to the left one, and the trace prints both chains after every step of `[1, 2, 3, 4]`.

In [ ]:
def trace_reverse(values):
    prev, cur = None, build_list(values)

    def show(step):
        done = " <- ".join(["None"] + [str(v) for v in reversed(to_list(prev))])
        rest = " -> ".join([str(v) for v in to_list(cur)] + ["None"])
        print(f"step {step}:  {done:<26}|  {rest}")

    show(0)
    step = 0
    while cur:
        nxt = cur.next
        cur.next = prev
        prev, cur = cur, nxt
        step += 1
        show(step)


trace_reverse([1, 2, 3, 4])

**Try it**
- Run `trace_reverse([7])` and `trace_reverse([])`: one step for one box, and for the empty list only the start line (`prev` stays `None`, which is the right answer).
- Count the boxes on every printed line: the two chains together always hold all n boxes, with no box in both. That is the invariant, visible: nothing is ever lost, it only changes sides.
- Change the loop to `while cur.next:` and run `trace_reverse([1, 2, 3])`: the last line still shows `3 -> None` on the right. The last box is never flipped, so you would return `2 -> 1` and lose the 3.

### Where it goes wrong

1. **Losing the rest of the list.** Writing an arrow before naming the box it pointed to: `cur.next = prev` before `nxt = cur.next` turns `[1, 2, 3]` into `[1]`. Fix: name every box the step touches, then rewire, then move.
2. **Returning the old head.** After a reversal `head` is the *tail*: returning it gives `[1]` for `[1, 2, 3]`. Return `prev` (reversal) or `dummy.next` (anything built behind a dummy).
3. **The head without a dummy.** Without the dummy and without a special case, `remove_nth_from_end` on `[1]` with n = 1 runs `fast` off the end and hits `None.next`, an `AttributeError`. A dummy gives every box a box before it; then return `dummy.next`.
4. **Off-by-one fingers.** A loop that only checks `fast.next` crashes on `[1, 2, 3, 4]` (guard with `while fast and fast.next:`); a gap of n instead of n + 1 deletes the wrong box (`[1, 2, 3, 4, 5], 2` → `[1, 2, 3, 4]`).
5. **Values instead of boxes.** Two different boxes can hold the same value. Meeting points and intersections compare boxes: `a is b`, never `a.val == b.val`. Two separate lists `[1]` and `[1]` share no box, but comparing values reports their first boxes as the intersection.
6. **Forgetting to cut.** When you split a list to reorder it, sort it or partition it, end the first part with `None`. Reordering `[1, 2, 3, 4]` without the cut leaves box 3 pointing at itself: `[1, 4, 2, 3, 3, 3, ...]`. Partitioning `[2, 1]` around 2, smaller values first, into two dummy chains without `big_tail.next = None` gives `[1, 2, 1, 2, ...]`. The palindrome check gets away without a cut only because it stops when the reversed half ends.
7. **Advancing after a delete.** After `cur.next = cur.next.next`, nobody has checked the new `cur.next`: move `cur` only when you did not delete. With an unconditional `cur = cur.next`, removing 2 from `[1, 2, 2, 3]` gives `[1, 2, 3]`.
8. **One-line multiple assignment.** Python evaluates the right side first, then assigns the targets left to right. `cur, cur.next, prev = cur.next, prev, cur` sets the `next` of the *new* `cur` and crashes on `[1, 2, 3]` (`AttributeError`); `cur.next, prev, cur = prev, cur, cur.next` works. If unsure, one assignment per line.
9. **Comparing before moving in Floyd.** `slow` and `fast` both start on the head, so `slow is fast` is true before the first step: compare first, and the loop-free list `[1, 2]` reports its head as the entrance of a cycle. Move first, then compare.

### Edge cases to say out loud

Empty list (`None`) · one box · two boxes (even length for fast/slow) · remove the head (n = length) · remove the last box (n = 1) · lists of different lengths · duplicate values (compare boxes, not values) · a cycle through the head, or a box pointing at itself.

In [ ]:
assert to_list(reverse_list(None)) == [] and to_list(reverse_list(build_list([7]))) == [7]
assert to_list(merge_lists(None, build_list([0]))) == [0]
assert to_list(merge_lists(build_list([5]), build_list([1, 2, 3]))) == [1, 2, 3, 5]
assert middle_node(build_list([1])).val == 1
assert middle_node(build_list([1, 2])).val == 2                        # second middle
assert to_list(remove_nth_from_end(build_list([1]), 1)) == []          # the only box
assert to_list(remove_nth_from_end(build_list([1, 2]), 2)) == [2]      # the head
assert to_list(remove_nth_from_end(build_list([1, 2]), 1)) == [1]      # the last box
assert to_list(swap_pairs(None)) == [] and to_list(swap_pairs(build_list([1]))) == [1]
assert to_list(remove_elements(build_list([6, 6, 1]), 6)) == [1]       # a run at the head
print("edge cases pass")

**Try it**
- Predict, then add: `assert middle_node(None) is None` (the `while` never runs, so `slow` is still `None`).
- What does `remove_nth_from_end(build_list([1, 2, 3]), 3)` return? Write the assert first: `[2, 3]`. When n equals the length you remove the head, the case the dummy exists for.
- Change `<=` to `<` in `merge_lists` and rerun: every assert still passes. On ties it only changes *which* of two equal boxes goes first (keeping `<=` keeps the merge stable).

### Variations

| Variation | What changes from the template | Problems |
|---|---|---|
| **Several arrows in one step** | name every box, rewire in any order, move the finger last | Swap Nodes in Pairs (24) |
| **Delete while walking** | move the finger only when nothing was deleted; for 82, drop the whole run of duplicates and keep `prev` where it is | Remove Linked List Elements (203); Remove Duplicates from Sorted List II (82): delete every value that appears more than once |
| **Cycle: is there one, where does it start?** | fast/slow race; after they meet, restart one finger at the head and step both by 1 | Linked List Cycle (141); Linked List Cycle II (142): the box where the loop begins |
| **Floyd on an array** | the "next" of index `i` is `nums[i]`; the duplicate value is the cycle entrance | Find the Duplicate Number (287): the repeated value among n + 1 numbers in `[1, n]` |
| **Fold in half** | middle → cut → reverse the back half → walk both halves together | Palindrome Linked List (234); Reorder List (143): first, last, second, second-to-last, ... |
| **Merge sort** | middle (fingers start on a dummy) → cut → sort both halves → merge | Sort List (148) |
| **Reverse a block** | reverse with `prev` starting at the box *after* the block, hook the box before it to the new head; in a loop with a look-ahead for k-groups | Reverse Linked List II (92): only positions left..right; Reverse Nodes in k-Group (25) |
| **Two chains, then join** | one dummy per chain, append each box to its chain, join them and cut the last tail | Partition List (86): smaller values first, both groups in order; Odd Even Linked List (328): odd positions first |
| **Rotate** | count n, close the list into a ring, cut after box n − k % n | Rotate List (61): shift every box k places to the right |
| **Two lists in lockstep** | one finger per list, step together (merge, add with a carry, line up two tails) | Merge Two Sorted Lists (21); Add Two Numbers (2): digits stored ones first; Intersection of Two Linked Lists (160) |
| **Copy with extra arrows** | dict old box → new box, then wire `next` and `random` through it | Copy List with Random Pointer (138) |
| **Recency order with O(1) moves** | dict + doubly linked list with two sentinels, in [Design Problems](11_Design.ipynb#s24) | LRU Cache (146) |

Cycles come first: only arrows can loop back on themselves, and a race between two fingers is how you notice. Linked List Cycle (141) asks whether the arrows ever loop back; Linked List Cycle II (142) asks for the box where the loop begins. In `3 → 2 → 0 → -4`, with the last box pointing back at the 2, the answer is the box that holds 2.

Inside a loop, `fast` gains exactly one box per step on `slow`, so the gap shrinks by one each step and must reach zero: they meet, and `fast` can't jump over `slow`. A speed-3 finger would not do: chasing a speed-1 finger round a 4-box loop from 1 apart, it sees the gap go 1, 3, 1, 3, ... forever. Without a loop, `fast` falls off the end.

Phase 2 finds the entrance because **whole laps don't move you**. After the meeting, restart one finger at the head and step both one box at a time: they meet exactly on the entrance.

```text
 head ------ a boxes ------> E ------ b boxes ------> M        E = entrance, M = where they met
                             ^                        |        c = boxes around the loop
                             +----- c - b boxes ------+

 when they meet, slow has walked a + b and fast 2(a + b); fast's extra a + b steps were whole
 laps, so a + b = k*c. Walk slow a more steps: it has walked a + k*c in total, which ends where
 a steps end, on E. A finger restarted at the head also reaches E after exactly a steps.
```

Find the Duplicate Number (287) asks for the one value that appears more than once among n + 1 numbers in `[1, n]`, without changing the array and with O(1) extra space: `[1, 3, 4, 2, 2] → 2`. Read index `i` as a box whose arrow points to index `nums[i]`. Values are in `[1, n]`, so no arrow enters index 0: the walk from 0 can never come back to 0, so it must run into a loop.

The box where it enters has two incoming arrows, one from the tail and one from the loop's last box. Two arrows into box v means v is stored twice, so the entrance is the duplicate.

```text
 nums  = [1, 3, 4, 2, 2]
 index:   0  1  2  3  4

 0 -> 1 -> 3 -> 2 -> 4           boxes 3 and 4 both point at 2: two arrows enter 2,
                ^    |           so 2 is the duplicate AND the entrance of the loop
                +----+
```

The cell runs both phases twice: `detect_cycle` on boxes made by `build_cycle`, whose last box links back to index `pos`, and `find_duplicate` on an array, where index `i` points to index `nums[i]` and `==` on two indices already means "the same box". The two asserts at the end are the edge cases: a box that points at itself, and a loop through the head.

In [ ]:
def build_cycle(values, pos):                # the last box links back to index pos (-1: no cycle)
    nodes = [ListNode(v) for v in values]
    for a, b in zip(nodes, nodes[1:]):
        a.next = b
    if pos >= 0:
        nodes[-1].next = nodes[pos]
    return nodes[0]


def detect_cycle(head):
    slow = fast = head
    while fast and fast.next:                # phase 1: race
        slow, fast = slow.next, fast.next.next
        if slow is fast:                     # met inside the loop (compare boxes, not values)
            break
    if fast is None or fast.next is None:    # the race ended because fast fell off: no cycle
        return None
    finger = head                            # phase 2: from the head and from the meeting point
    while finger is not slow:
        finger, slow = finger.next, slow.next
    return finger                            # the entrance


def find_duplicate(nums):                    # index i points to index nums[i]
    slow = fast = 0                          # 0 is never a target: it starts the tail
    while True:
        slow, fast = nums[slow], nums[nums[fast]]
        if slow == fast:                     # indices, so == is "same box" here
            break
    finger = 0
    while finger != slow:
        finger, slow = nums[finger], nums[slow]
    return finger                            # the box with two incoming arrows


print(detect_cycle(build_cycle([3, 2, 0, -4], 1)).val, detect_cycle(build_cycle([1, 2], -1)))   # 2 None
print(find_duplicate([1, 3, 4, 2, 2]), find_duplicate([3, 1, 3, 4, 2]))                          # 2 3
assert detect_cycle(build_cycle([1], 0)).val == 1                       # a box pointing at itself
assert detect_cycle(build_cycle([1, 2], 0)).val == 1                    # a loop through the head

**Try it**
- Return `slow` right after phase 1 (skip phase 2) and rerun the first example: you get `-4`, the box where the race happened to end, not the entrance `2`.
- Check the arithmetic on a bigger rho: in `build_cycle(list(range(10)), 3)` the tail has a = 3 boxes and the loop c = 7. Print `slow.val` after phase 1: `7`, so b = 4 and a + b = 7 is exactly one lap.
- Compare before moving: put the `if slow is fast: break` lines *above* the step line in phase 1. The cell now prints `3` and a `ListNode` instead of `2 None`: the fingers "meet" on the head before taking a single step.
- Predict `find_duplicate([2, 5, 9, 6, 9, 3, 8, 9, 7, 1])` before running (9: it appears three times, so three arrows enter box 9).

Folding in half chains the templates: find the middle, cut, reverse, walk. Palindrome Linked List (234) asks whether the values read the same both ways, `[1, 2, 2, 1]` → True. Reorder List (143) interleaves the front and the back in place, `[1, 2, 3, 4, 5]` → `[1, 5, 2, 4, 3]`. Both need the back half walked backwards. You can't walk backwards, but you can *make* the back half run backwards by reversing it in place.

```text
 1 -> 2 -> 3 -> 4 -> 5         slow stops on 3, the last box of the first half
 1 -> 2 -> 3     5 -> 4        cut after 3 and reverse the back half
 1 -> 5 -> 2 -> 4 -> 3         zip: one box from the left half, one from the right
```

Each of them starts by finding the middle, and the only decision is which middle. With the guard `while fast and fast.next`, fingers that start on `head` stop on the 3 of both `[1, 2, 3, 4]` and `[1, 2, 3, 4, 5]`; on the even length that is the first box of the back half, which is what the palindrome check needs.

Fingers that start on a dummy before `head` stop on the 2 of `[1, 2, 3, 4]` and on the 3 of `[1, 2, 3, 4, 5]`: the box *before* the back half, the only box that can cut it off.

Reorder needs that box, and so does Sort List (148), which sorts a list in O(n log n) time, `[4, 2, 1, 3]` → `[1, 2, 3, 4]`, by merge sort: find the middle, cut, sort both halves, and merge them with `merge_lists`. Both start their fingers on the dummy, the same rule as n-th from the end, while the palindrome check reuses `middle_node`.

In [ ]:
def reorder_list(head):                      # L0 -> Ln -> L1 -> Ln-1 -> ...  in place
    slow = fast = ListNode(0, head)          # start on a dummy: slow ends BEFORE the back half
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    second = slow.next
    slow.next = None                         # cut: the halves must not touch
    second = reverse_list(second)            # the back half, now running backwards
    first = head
    while second:                            # the first half is as long or one longer
        n1, n2 = first.next, second.next     # name both rests before rewiring
        first.next, second.next = second, n1
        first, second = n1, n2


def is_palindrome_list(head):
    second = reverse_list(middle_node(head)) # back half reversed (it includes the middle if odd)
    first = head
    while second:                            # the reversed half is never the longer one
        if first.val != second.val:
            return False
        first, second = first.next, second.next
    return True


def sort_list(head):                         # merge sort: middle, cut, sort both halves, merge
    if head is None or head.next is None:
        return head
    slow = fast = ListNode(0, head)          # start on a dummy: slow stops BEFORE the back half
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    second = slow.next
    slow.next = None                         # cut
    return merge_lists(sort_list(head), sort_list(second))


h = build_list([1, 2, 3, 4, 5]); reorder_list(h); print(to_list(h))      # [1, 5, 2, 4, 3]
h = build_list([1, 2, 3, 4]); reorder_list(h); print(to_list(h))         # [1, 4, 2, 3]
print(is_palindrome_list(build_list([1, 2, 2, 1])), is_palindrome_list(build_list([1, 2])))   # True False
print(to_list(sort_list(build_list([4, 2, 1, 3]))), to_list(sort_list(build_list([-1, 5, 3, 4, 0]))))   # [1, 2, 3, 4] [-1, 0, 3, 4, 5]

**Try it**
- Delete the cut `slow.next = None` in `reorder_list` and rerun: both reordered lists now run on until `to_list`'s limit. `to_list(h, 8)` shows `[1, 4, 2, 3, 3, 3, 3, 3]`: box 3 points at itself.
- Split `first.next, second.next = second, n1` into two lines: either order still gives `[1, 5, 2, 4, 3]`, because `n1` and `n2` already name every box. Then drop `n2` and advance with `second = second.next` *after* the rewire: `[1, 5, 2, 3]`. The 4 is lost, because `second.next` was rewired before you read it.
- In `sort_list`, start the fingers on `head` instead of a dummy: `sort_list(build_list([2, 1]))` raises `RecursionError`. `slow` stops on the second box, so the back half is empty and the front half is the whole list again, forever.
- Run `h = build_list([1, 2, 3, 2, 1]); print(is_palindrome_list(h), to_list(h))`: `True [1, 2, 3]`. The check rewired the input; reverse the back half again if the caller needs the list intact.

Reversal returns for one part of the list. Reverse Linked List II (92) reverses only positions `left..right`, counting from 1: `[1, 2, 3, 4, 5]` with 2..4 → `[1, 4, 3, 2, 5]`. It is the plain reversal with two extra fingers: `before`, the box before the block, and `after`, the first box after it. Start the reversal with `prev = after`, so the block's new tail already points onward, then hook `before` to the block's new head.

```text
 left = 2, right = 4:   D -> 1 -> 2 -> 3 -> 4 -> 5          before = 1, after = 5
                        D -> 1    4 -> 3 -> 2 -> 5          reversed with prev starting at 5
                        D -> 1 -> 4 -> 3 -> 2 -> 5          before.next = 4: hooked in
```

Reverse Nodes in k-Group (25) reverses every full block of k boxes and leaves a shorter rest alone: `[1, 2, 3, 4, 5]` with k = 2 → `[2, 1, 4, 3, 5]`, and with k = 3 → `[3, 2, 1, 4, 5]`. It is the same block reversal in a loop, with a look-ahead that stops when fewer than k boxes are left; each block's old first box, now its tail, becomes the next `before`.

In [ ]:
def reverse_block(before, after):            # reverse the boxes strictly between before and after
    prev, cur = after, before.next           # prev = after: the block's new tail points on
    while cur is not after:
        nxt = cur.next
        cur.next = prev
        prev, cur = cur, nxt
    first = before.next                      # the old first box is now the block's tail
    before.next = prev                       # hook the block's new head in
    return first


def reverse_between(head, left, right):      # 92: reverse positions left..right (1-based)
    dummy = ListNode(0, head)
    before = dummy
    for _ in range(left - 1):
        before = before.next                 # the box before the block
    after = before.next
    for _ in range(right - left + 1):
        after = after.next                   # the first box after the block
    reverse_block(before, after)
    return dummy.next


def reverse_k_group(head, k):                # 25: the same block reversal, in a loop
    dummy = ListNode(0, head)
    before = dummy
    while True:
        last = before                        # look ahead: are there k more boxes?
        for _ in range(k):
            last = last.next
            if last is None:
                return dummy.next            # fewer than k left: they stay as they are
        before = reverse_block(before, last.next)   # the block's tail is the next "before"


print(to_list(reverse_between(build_list([1, 2, 3, 4, 5]), 2, 4)))   # [1, 4, 3, 2, 5]
print(to_list(reverse_k_group(build_list([1, 2, 3, 4, 5]), 2)))      # [2, 1, 4, 3, 5]
print(to_list(reverse_k_group(build_list([1, 2, 3, 4, 5]), 3)))      # [3, 2, 1, 4, 5]

**Try it**
- In `reverse_block`, start with `prev = None` instead of `after`: the 92 example prints `[1, 4, 3, 2]`. Box 2 now points at `None`, so the 5 is cut off.
- `reverse_between(build_list([1, 2, 3]), 1, 3)` reverses everything: `before` is the dummy, so position 1 needs no special case.
- Predict before running: k = 1 returns the list unchanged, k = 5 reverses all of it, k = 6 changes nothing (there is no full block).
- Print `to_list(dummy.next)` right after the `before = reverse_block(...)` line: for k = 2 you see `[2, 1, 3, 4, 5]` and then `[2, 1, 4, 3, 5]`, one finished block per line.

Two lists can also be walked in lockstep, one finger per list. Add Two Numbers (2) adds two numbers stored as lists of digits, ones digit first: `2 → 4 → 3` plus `5 → 6 → 4` is 342 + 465 = 807, stored as `7 → 0 → 8`. It is schoolbook addition: walk both lists, write `total % 10`, carry `total // 10`, and keep going while either list or the carry remains.

Intersection of Two Linked Lists (160) asks for the first box two lists share, or `None`. Past the merge box both lists are the *same* boxes, so fingers that start the same distance from the end reach it together. Count the lengths and skip the longer list's extra prefix, or let each finger walk its own list and then the other: both then walk A's own part, B's own part and the shared tail.

Lowest Common Ancestor of a Binary Tree III (1650), where every tree node knows its parent, is the same problem: walk up from p and from q, and the two upward paths merge at the ancestor.

```text
 A:      4 -> 1 \                 pa walks 4 1 8 4 5, then B's own part 5 6 1   (8 boxes)
                 8 -> 4 -> 5      pb walks 5 6 1 8 4 5, then A's own part 4 1   (8 boxes)
 B: 5 -> 6 -> 1 /                 so the next step puts both on the shared 8
```

The cell adds 342 + 465 and 99 + 1, whose final carry needs a box of its own, and builds the picture's two lists around the shared tail `8 → 4 → 5`.

In [ ]:
def add_two_numbers(l1, l2):                 # ones digit first: 342 is stored as 2 -> 4 -> 3
    dummy = tail = ListNode()
    carry = 0
    while l1 or l2 or carry:                 # a leftover carry still makes a box
        total = carry
        if l1:
            total, l1 = total + l1.val, l1.next
        if l2:
            total, l2 = total + l2.val, l2.next
        carry, digit = divmod(total, 10)
        tail.next = ListNode(digit)          # RECORD: one new box per column
        tail = tail.next
    return dummy.next


def intersection_node(a, b):
    pa, pb = a, b                            # each finger walks its own list, then the other
    while pa is not pb:                      # boxes, not values
        pa = pa.next if pa else b            # fell off A: continue at the head of B
        pb = pb.next if pb else a
    return pa                                # the shared box, or None (both fell off together)


print(to_list(add_two_numbers(build_list([2, 4, 3]), build_list([5, 6, 4]))))   # [7, 0, 8]  (342 + 465 = 807)
print(to_list(add_two_numbers(build_list([9, 9]), build_list([1]))))             # [0, 0, 1]  (99 + 1 = 100)
shared = build_list([8, 4, 5])
a, b = ListNode(4, ListNode(1, shared)), ListNode(5, ListNode(6, ListNode(1, shared)))
print(intersection_node(a, b).val, intersection_node(build_list([1]), build_list([1])))   # 8 None

**Try it**
- Change `while l1 or l2 or carry:` to `while l1 or l2:` and rerun 99 + 1: `[0, 0]`. The final carry never got its box.
- In `intersection_node`, compare values (`while pa.val != pb.val:`) and rerun: `AttributeError` as soon as `pa` falls off A (`None.val`). With `is`, `None` is a legal place to stop, and two boxes that merely hold equal values (the two 1s) never count as the same box.
- Print `pa.val if pa else None, pb.val if pb else None` at the top of the loop, then call `intersection_node(a, b)`: nine lines, ending with `1 1` (two different boxes holding equal values); the next step puts both fingers on the shared 8.
- The counting version lines up the same way: measure both lengths, move the longer list's finger ahead by the difference, then step both until `a is b`. Write it and check that it also gives 8.

The last variation copies a list that has two arrows per box. Copy List with Random Pointer (138) asks for a deep copy of a list whose boxes also carry a `random` arrow to any box or to `None`: every arrow of the copy must point into the copy, never back into the original. A deep copy needs a translator from "a box of the old list" to "its copy".

Pass 1 creates a copy of every box and stores the pair in a dict; pass 2 wires each copy's `next` and `random` through that dict. The dict starts as `{None: None}`, so an arrow to nothing translates to nothing instead of raising `KeyError`. The O(1)-space version weaves each copy right after its original, `A → A' → B → B'`, so that "the copy of X" is simply `X.next`, and then unweaves the two lists.

<details><summary>The two-pass copy, with a test that checks every arrow</summary>

```py
class RandomNode:
    def __init__(self, val, next=None, random=None):
        self.val, self.next, self.random = val, next, random


def copy_random_list(head):
    old_to_new = {None: None}                # "points at nothing" translates to nothing
    node = head
    while node:                              # pass 1: one new box per old box
        old_to_new[node] = RandomNode(node.val)
        node = node.next
    node = head
    while node:                              # pass 2: wire the copies through the dict
        old_to_new[node].next = old_to_new[node.next]
        old_to_new[node].random = old_to_new[node.random]
        node = node.next
    return old_to_new[head]


def build_random(pairs):                     # [(val, index the random arrow points to), ...]
    nodes = [RandomNode(v) for v, _ in pairs]
    for i, (_, r) in enumerate(pairs):
        nodes[i].next = nodes[i + 1] if i + 1 < len(nodes) else None
        nodes[i].random = nodes[r] if r is not None else None
    return nodes[0] if nodes else None


def describe(head):                          # [(val, index of the random target), ...]
    boxes = []
    while head:
        boxes.append(head)
        head = head.next
    return [(n.val, None if n.random is None else boxes.index(n.random)) for n in boxes]


orig = build_random([(7, None), (13, 0), (11, 4), (10, 2), (1, 0)])
clone = copy_random_list(orig)
print(describe(clone))                       # [(7, None), (13, 0), (11, 4), (10, 2), (1, 0)]
print(clone is orig, clone.next.random is orig, clone.next.random is clone)   # False False True
```

</details>

### Say it in the interview

> "Copying into an array costs O(n) extra space, so I'll rewire the nodes in place. A dummy before the head makes the head an ordinary node. In each step I first name every node I'm about to touch, then change the arrows, then move my pointers. Invariant: everything before `cur` is finished and hangs off the dummy; everything from `cur` on is untouched. One pass: O(n) time, O(1) space. Let me check `[]`, `[1]` and `[1, 2]` by hand."

While coding, point at the naming line ("so I don't lose the rest"), at `return dummy.next` ("the head may have changed"), and for fast/slow say the loop guard out loud ("fast can take two more steps"). Be ready for the follow-ups:

- *Recursive version?* Trust the call to reverse the rest, then `head.next.next = head; head.next = None`, at the price of O(n) call stack, about 1000 boxes in Python.
- *Must the input stay intact (palindrome)?* Reverse the back half again before returning.
- *Floyd: why do the fingers meet, and why does phase 2 find the entrance?* The gap closes by exactly one per step, so it can't be skipped; whole laps don't move you, so a steps from the meeting point land where a steps from the head land.
- *Remove the n-th from the end in one pass?* A gap of n + 1 between two fingers that start on the dummy.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Add Two Numbers | `linked_list/add_two_numbers.py` | ones digit first, so add column by column with a carry; loop while either list or the carry remains |
| Copy List with Random Pointer | `linked_list/copy_list_with_random_pointer.py` | dict old box → new box (or weave copies after originals), then wire `next`/`random` through it |
| Find the Duplicate Number | `linked_list/find_the_duplicate_number.py` | read `i → nums[i]` as a list from index 0; the duplicate has two incoming arrows = the cycle entrance (Floyd) |
| Intersection of Two Linked Lists | `linked_list/intersection_of_two_linked_lists.py` | skip the longer list's extra prefix, then walk both until `a is b` |
| Linked List Cycle | `linked_list/linked_list_cycle.py` | fast gains one box per step on slow inside a loop; meeting = cycle, falling off = none |
| Linked List Cycle II | `linked_list/linked_list_cycle_ii.py` · `practice/simple/23_linked_list_cycle_ii.py` | whole laps don't move you: after the meeting, a finger from the head and one from the meeting point meet at the entrance |
| Merge Two Sorted Lists | `linked_list/merge_two_sorted_lists.py` · `practice/simple/22_merge_two_sorted_lists.py` | dummy + tail: always hang the smaller front box; attach the leftover whole |
| Palindrome Linked List | `linked_list/palindrome_linked_list.py` | find the middle, reverse the back half in place, compare the halves |
| Remove Nth Node From End of List | `linked_list/remove_nth_from_end.py` · `practice/simple/25_remove_nth_node_from_end.py` | gap of n + 1 from a dummy: when fast falls off, slow is just before the victim |
| Reorder List | `linked_list/reorder_list.py` | middle (start on a dummy), cut, reverse the back half, zip the two halves |
| Reverse Linked List | `linked_list/reverse_linked_list.py` · `practice/simple/21_reverse_linked_list.py` | name next, flip the arrow, move both fingers; `prev` ends as the new head |
| Reverse Nodes in k-Group | `linked_list/reverse_nodes_in_k_group.py` | look ahead k boxes; reverse with `prev` = the box after the block; hook the box before it to the new head |

### Self-check

1. In Linked List Cycle II, why does a finger restarted at the head meet the other finger exactly at the entrance?
<details><summary>Answer</summary>With a tail of a boxes, a loop of c boxes and a meeting point b boxes into the loop, slow walked a + b and fast 2(a + b). Fast's extra a + b steps were whole laps, so a + b = k·c. Walking a more steps from the meeting point brings slow to a + k·c steps in total, which ends where a steps end: the entrance, where the finger from the head also arrives after a steps.</details>

2. In Remove Linked List Elements, why doesn't `cur` move after a delete?
<details><summary>Answer</summary>The delete pulls a new box into <code>cur.next</code>, and nobody has checked it yet. Moving on would skip it, so a run of equal values (<code>[1, 2, 2, 3]</code>, remove 2) keeps one copy. Move only when you keep the box.</details>

3. In Find the Duplicate Number, why can index 0 never be inside the cycle, and why is the cycle's entrance the duplicate?
<details><summary>Answer</summary>Every value is in [1, n], so no arrow points at index 0: the walk leaves it and never returns, so it is the start of the tail. The entrance is the one box with two incoming arrows (one from the tail, one from inside the loop), and two arrows into box v means two indices hold the value v.</details>